# CS781 Phase 3 - Rhetorical Role prediction on the Phase-1 LegalLM

**Inputs to attach** (Add Input): your Phase-1 model folder (`config.json`, `pytorch_model.bin`, `tokenizer.json`), `evaluate_phase3.py`,
and the Phase-3 data (if the data is not attached it is downloaded from `Exploration-Lab/CS-781-Capstone/Phase_3_RR` with your `capstoned` Kaggle secret, as in Phase 2).
Accelerator: **GPU T4 x2**.

**Run order:** 1 paths -> 2 write code -> 3 inspect data (stop and check) -> 4 smoke tests -> 5 round 1 -> 6 compare -> 7 round 2 -> 8 downloads.

| Run | What changes | Question it answers |
|---|---|---|
| `e0_sent_full` | each sentence encoded **alone**, no document context (control) | baseline: full fine-tuning, plain sentence classifier |
| `e1_ctx_full` | sentences **packed** into 1024-token windows, bidirectional, + doc-position + domain embeddings + BiGRU | how much does document context add, on the same model and optimiser? |
| `e2_ctx_lorma` | round 2: `e1` pipeline with LoRMA instead of full FT | does Phase 2's PEFT win (0.54 vs 0.46) carry over? |
| `e2_ctx_full_sqrtw` | round 2: `e1` + sqrt inverse-frequency class weights | do rare roles improve without hurting the rest? |

Every run: its own folder under `/kaggle/working/rr/<name>/`, selection on **your official evaluator's** dev Macro-F1, `best.pt` + `last.pt` (auto-resume), `dev_report_best.json` (per-class / per-domain / confusion), `submission_<name>.zip` with `prediction.json` at the root. Finished runs are never overwritten.

## 1. Paths and hardware

In [1]:
import os, json, glob, subprocess, sys, time

DATA_DIR = None           # folder that holds the Phase-3 files (None = search /kaggle/input, else download)
PHASE1_MODEL_DIR = None   # folder with config.json + pytorch_model.bin + tokenizer.json
EVALUATOR = None          # path to YOUR evaluate_phase3.py (None = search /kaggle/input)
OUT_ROOT = "/kaggle/working/rr"
CODE_DIR = "/kaggle/working/rr_code"
HF_DATASET_REPO, HF_PREFIX, HF_SECRET_NAME = "Exploration-Lab/CS-781-Capstone", "Phase_3_RR", "capstoned"

def _walk(root="/kaggle/input"):
    for dp, dn, fn in os.walk(root):
        yield dp, dn, fn

def find_data_dir():
    for dp, dn, fn in _walk():
        if os.path.basename(dp).lower() in ("phase_3_rr", "phase3_rr", "phase-3-rr"):
            return dp
    print("Phase-3 data not attached -> downloading from Hugging Face")
    from huggingface_hub import snapshot_download
    token = os.environ.get("HF_TOKEN")
    if not token:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret(HF_SECRET_NAME)
    local = snapshot_download(HF_DATASET_REPO, repo_type="dataset", allow_patterns=[HF_PREFIX + "/*"],
                              local_dir="/kaggle/working/hf_phase3", token=token)
    return os.path.join(local, HF_PREFIX)

def find_legallm():
    hits = []
    for dp, dn, fn in _walk():
        if {"config.json", "pytorch_model.bin"} <= set(fn):
            try:
                c = json.load(open(os.path.join(dp, "config.json")))
            except Exception:
                continue
            if {"n_layer", "d_model", "vocab_size", "max_seq_len"} <= set(c):
                hits.append(dp)
    if not hits:
        raise FileNotFoundError("Phase-1 LegalLM folder (config.json + pytorch_model.bin) not found under /kaggle/input")
    hits.sort(key=lambda p: ("legal" not in p.lower(), p))
    print("LegalLM folders found:", hits)
    return hits[0]

def find_file(name, root="/kaggle/input"):
    hits = sorted(os.path.join(dp, name) for dp, dn, fn in _walk(root) if name in fn)
    if not hits:
        raise FileNotFoundError(f"{name} not found under {root} - attach it or set the path above")
    if len(hits) > 1:
        print(f"several {name} found, using the first:", hits)
    return hits[0]

DATA_DIR = DATA_DIR or find_data_dir()
PHASE1_MODEL_DIR = PHASE1_MODEL_DIR or find_legallm()
EVALUATOR = EVALUATOR or find_file("evaluate_phase3.py")
TOKENIZER = os.path.join(PHASE1_MODEL_DIR, "tokenizer.json")
if not os.path.exists(TOKENIZER):
    TOKENIZER = find_file("tokenizer.json")
os.makedirs(OUT_ROOT, exist_ok=True); os.makedirs(CODE_DIR, exist_ok=True)
PATHS = dict(data_dir=DATA_DIR, model_dir=PHASE1_MODEL_DIR, tokenizer=TOKENIZER, evaluator=EVALUATOR, out_root=OUT_ROOT)
json.dump(PATHS, open("/kaggle/working/rr_paths.json", "w"), indent=1)
print(json.dumps(PATHS, indent=1))
print("pytorch_model.bin size: %.1f MB" % (os.path.getsize(os.path.join(PHASE1_MODEL_DIR, "pytorch_model.bin")) / 2**20))
print(open(os.path.join(PHASE1_MODEL_DIR, "config.json")).read())
import torch
print("torch", torch.__version__, "| GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
for i in range(torch.cuda.device_count()):
    print(f"  cuda:{i} {torch.cuda.get_device_properties(i).total_memory / 2**30:.1f} GB")

Phase-3 data not attached -> downloading from Hugging Face


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

LegalLM folders found: ['/kaggle/input/datasets/adyansubeura/model-capstone']
{
 "data_dir": "/kaggle/working/hf_phase3/Phase_3_RR",
 "model_dir": "/kaggle/input/datasets/adyansubeura/model-capstone",
 "tokenizer": "/kaggle/input/datasets/adyansubeura/model-capstone/tokenizer.json",
 "evaluator": "/kaggle/input/datasets/adyansubeura/model-capstone/evaluate_phase3.py",
 "out_root": "/kaggle/working/rr"
}
pytorch_model.bin size: 417.9 MB
{
  "vocab_size": 32000,
  "n_layer": 12,
  "n_head": 12,
  "d_model": 768,
  "d_ff": 2048,
  "max_seq_len": 1024,
  "rope_theta": 10000.0,
  "norm_eps": 1e-05
}
torch 2.11.0+cu128 | GPUs: ['Tesla T4', 'Tesla T4']
  cuda:0 14.6 GB
  cuda:1 14.6 GB


## 2. Write the pipeline code (`rr_data.py`: format detection; `rr_train.py`: model, training, evaluation, submission)

In [2]:
%%writefile /kaggle/working/rr_code/rr_data.py
"""CS781 Phase 3 (Rhetorical Roles) - data discovery, format detection and validation.

The exact file layout of Phase_3_RR/ is detected at runtime, never assumed. Supported shapes
(JSON, JSONL, CSV/TSV), per split (train / dev / test) and domain (CL / IT):
  * {doc_id: {"sentences": [...], "labels": [...]}}             (any common key names)
  * [{"id": ..., "sentences": [...], "labels": [...]}, ...]
  * {doc_id: [sentence, ...]}  +  {doc_id: [label, ...]}        (text and truth in separate files)
  * {doc_id: [{"text"/"sentence": ..., "label": ...}, ...]}
  * legal-eval style [{"id", "annotations": [{"result": [{"value": {"text", "labels": [...]}}]}]}]
  * CSV/TSV rows with a document-id column, a sentence column and (optionally) a label column
Split and domain come from the file path (e.g. .../CL/train.json, it_dev.json) or a record field.
Anything unrecognised stops with an explicit message instead of guessing.
"""
import csv
import json
import os
import random
import re
from collections import Counter, defaultdict

LABELS = ["Fact", "Issue", "ArgumentPetitioner", "ArgumentRespondent", "Statute", "Dissent",
          "PrecedentReliedUpon", "PrecedentNotReliedUpon", "PrecedentOverruled", "RulingByLowerCourt",
          "RatioOfTheDecision", "RulingByPresentCourt", "None"]
LABEL_SET = set(LABELS)
DOMAINS = ["CL", "IT"]
EXPECTED_SENTENCES = {("train", "CL"): 10593, ("train", "IT"): 6252, ("dev", "CL"): 1396,
                      ("dev", "IT"): 746, ("test", "CL"): 1339, ("test", "IT"): 858}

ID_KEYS = ("id", "doc_id", "docid", "document_id", "documentid", "doc", "name", "file", "filename",
           "file_name", "case_id", "caseid", "doc_name")
SENT_KEYS = ("sentences", "sents", "sentence", "texts", "text", "segments", "lines")
LABEL_KEYS = ("labels", "label", "roles", "role", "rhetorical_roles", "rhetorical_role", "rr", "tags",
              "gold", "y")
DOMAIN_KEYS = ("domain", "category", "area", "subset")
CONTAINER_KEYS = ("data", "documents", "docs", "items", "records", "examples")


class Phase3FormatError(RuntimeError):
    pass


def _tokens(path):
    return [t for t in re.split(r"[^a-z0-9]+", path.lower()) if t]


def split_from_path(path):
    toks = _tokens(path)
    hits = set()
    for t in toks:
        if t.startswith("train"):
            hits.add("train")
        elif t in ("dev", "val", "valid", "validation", "development") or t.startswith("dev"):
            hits.add("dev")
        elif t.startswith("test"):
            hits.add("test")
    return hits.pop() if len(hits) == 1 else None


def domain_from_text(s):
    toks = _tokens(str(s))
    hits = set()
    for t in toks:
        if t in ("cl", "corporate", "corporatelaw", "company") or t.startswith("cl") and t[2:].isdigit():
            hits.add("CL")
        elif t in ("it", "infotech", "informationtechnology") or t.startswith("it") and t[2:].isdigit():
            hits.add("IT")
    return hits.pop() if len(hits) == 1 else None


def _first(d, keys):
    low = {k.lower(): k for k in d}
    for k in keys:
        if k in low:
            return d[low[k]]
    return None


def _is_label_list(v):
    return isinstance(v, list) and v and all(isinstance(x, str) for x in v) and all(x in LABEL_SET for x in v)


def _norm_label(x):
    if isinstance(x, list) and len(x) == 1:
        x = x[0]
    if isinstance(x, dict):
        x = _first(x, LABEL_KEYS)
        return _norm_label(x)
    return x


def _parse_sentence_dicts(items):
    sents, labs = [], []
    for it in items:
        s = _first(it, ("text", "sentence", "sent", "content", "line"))
        if s is None and "value" in it and isinstance(it["value"], dict):         # legal-eval result item
            s = it["value"].get("text")
            labs.append(_norm_label(it["value"].get("labels")))
        else:
            labs.append(_norm_label(_first(it, LABEL_KEYS)))
        sents.append(s)
    labs = None if all(l is None for l in labs) else labs
    return sents, labs


def parse_record(r, default_id=None):
    """-> (doc_id, sentences or None, labels or None, domain or None)"""
    did = _first(r, ID_KEYS)
    did = default_id if did is None else did
    dom = _first(r, DOMAIN_KEYS)
    dom = domain_from_text(dom) if dom is not None else None
    sents = labs = None
    ann = r.get("annotations")
    if isinstance(ann, list) and ann and isinstance(ann[0], dict) and isinstance(ann[0].get("result"), list):
        sents, labs = _parse_sentence_dicts(ann[0]["result"])
    else:
        sv = _first(r, SENT_KEYS)
        if isinstance(sv, dict):                                      # e.g. {"text": {...}} nested
            return parse_record({**sv, **{k: v for k, v in r.items() if k not in SENT_KEYS}}, did)
        if isinstance(sv, list) and sv and all(isinstance(x, dict) for x in sv):
            sents, labs = _parse_sentence_dicts(sv)
        elif isinstance(sv, list) and all(isinstance(x, str) for x in sv):
            sents = sv
        lv = _first(r, LABEL_KEYS)
        if isinstance(lv, list):
            labs = [_norm_label(x) for x in lv]
    if sents is not None and _is_label_list(sents) and labs is None:      # a truth file keyed under "text"
        sents, labs = None, sents
    return (None if did is None else str(did)), sents, labs, dom


def parse_obj(obj):
    out = []
    if isinstance(obj, dict):
        cont = _first(obj, CONTAINER_KEYS)
        if isinstance(cont, list) and len(obj) <= 3:
            return parse_obj(cont)
        for k, v in obj.items():
            if isinstance(v, list) and all(isinstance(x, str) for x in v):
                out.append((str(k), None, v, None) if _is_label_list(v) else (str(k), v, None, None))
            elif isinstance(v, list) and v and all(isinstance(x, dict) for x in v):
                s, l = _parse_sentence_dicts(v)
                out.append((str(k), s, l, None))
            elif isinstance(v, dict):
                out.append(parse_record(v, default_id=k))
            elif isinstance(v, list) and not v:
                out.append((str(k), [], None, None))
            else:
                raise Phase3FormatError(f"unrecognised value type for key {k!r}: {type(v).__name__}")
    elif isinstance(obj, list):
        for i, r in enumerate(obj):
            if not isinstance(r, dict):
                raise Phase3FormatError(f"list item {i} is {type(r).__name__}, expected an object")
            out.append(parse_record(r))
    else:
        raise Phase3FormatError(f"top-level JSON is {type(obj).__name__}")
    return out


def parse_csv(path):
    with open(path, encoding="utf-8", newline="") as f:
        sample = f.read(4096)
        f.seek(0)
        dialect = csv.Sniffer().sniff(sample, delimiters=",\t;|")
        rows = list(csv.DictReader(f, dialect=dialect))
    if not rows:
        return []
    cols = {c.lower(): c for c in rows[0]}
    idc = next((cols[k] for k in ID_KEYS if k in cols), None)
    sc = next((cols[k] for k in ("sentence", "text", "sent", "sentences") if k in cols), None)
    lc = next((cols[k] for k in LABEL_KEYS if k in cols), None)
    if idc is None or sc is None:
        raise Phase3FormatError(f"{path}: CSV needs a document-id and a sentence column, got {list(rows[0])}")
    docs = defaultdict(lambda: ([], []))
    order = []
    for r in rows:
        d = str(r[idc])
        if d not in docs:
            order.append(d)
        docs[d][0].append(r[sc])
        docs[d][1].append(r[lc] if lc else None)
    return [(d, docs[d][0], docs[d][1] if lc else None, None) for d in order]


def read_any(path):
    ext = os.path.splitext(path)[1].lower()
    if ext == ".jsonl":
        recs = [json.loads(l) for l in open(path, encoding="utf-8") if l.strip()]
        return parse_obj(recs)
    if ext == ".json":
        return parse_obj(json.load(open(path, encoding="utf-8")))
    if ext in (".csv", ".tsv"):
        return parse_csv(path)
    return None


def find_label_file(data_dir):
    for dp, _, fn in os.walk(data_dir):
        for f in fn:
            if "label" in f.lower() and os.path.splitext(f)[1].lower() in (".json", ".txt"):
                p = os.path.join(dp, f)
                try:
                    if f.lower().endswith(".txt"):
                        labs = [x.strip() for x in open(p, encoding="utf-8") if x.strip()]
                    else:
                        d = json.load(open(p, encoding="utf-8"))
                        if isinstance(d, list):
                            labs = d
                        elif isinstance(d, dict) and all(isinstance(v, int) for v in d.values()):
                            labs = list(d)
                        elif isinstance(d, dict):
                            labs = list(d.values())
                        else:
                            continue
                    if len(labs) == 13 and all(isinstance(x, str) for x in labs):
                        return p, labs
                except Exception:
                    continue
    return None, None


def inventory(data_dir, max_preview=300):
    """Human-readable listing of the dataset folder (for the notebook's inspection cell)."""
    lines = []
    for dp, _, fn in sorted(os.walk(data_dir)):
        for f in sorted(fn):
            p = os.path.join(dp, f)
            rel = os.path.relpath(p, data_dir)
            lines.append(f"{rel}  ({os.path.getsize(p) / 1024:.1f} KB)  split={split_from_path(rel)} "
                         f"domain={domain_from_text(rel)}")
            if f.lower().endswith((".json", ".jsonl", ".csv", ".tsv", ".txt")):
                with open(p, encoding="utf-8", errors="replace") as fh:
                    head = fh.read(max_preview).replace("\n", "\\n")
                lines.append(f"    preview: {head}")
    return "\n".join(lines)


def load_phase3(data_dir, seed=42, dev_docs_per_domain_fallback=5, verbose=True):
    """-> docs: list of dict(id, split, domain, sentences, labels|None), label_file_path|None"""
    label_path, file_labels = find_label_file(data_dir)
    if file_labels is not None and set(file_labels) != LABEL_SET:
        raise Phase3FormatError(f"label file {label_path} differs from the official 13 labels: {file_labels}")

    merged = {}                                      # (split, domain, id) -> dict
    used_files = []
    for dp, _, fn in sorted(os.walk(data_dir)):
        for f in sorted(fn):
            p = os.path.join(dp, f)
            if p == label_path:
                continue
            rel = os.path.relpath(p, data_dir)
            try:
                parsed = read_any(p)
            except Phase3FormatError as e:
                raise Phase3FormatError(f"{rel}: {e}")
            except (json.JSONDecodeError, UnicodeDecodeError, csv.Error) as e:
                if verbose:
                    print(f"  skip {rel}: not parseable ({type(e).__name__})")
                continue
            if parsed is None:
                continue
            split = split_from_path(rel)
            fdom = domain_from_text(rel)
            if split is None:
                print(f"  skip {rel}: cannot tell train/dev/test from its path")
                continue
            used_files.append(rel)
            for did, sents, labs, rdom in parsed:
                dom = rdom or fdom
                if did is None:
                    raise Phase3FormatError(f"{rel}: record without a document id")
                if dom is None:
                    raise Phase3FormatError(f"{rel}: cannot tell CL/IT for doc {did} (no domain in path or record)")
                key = (split, dom, did)
                cur = merged.setdefault(key, {"id": did, "split": split, "domain": dom,
                                              "sentences": None, "labels": None, "src": []})
                cur["src"].append(rel)
                if sents is not None:
                    if cur["sentences"] is not None and cur["sentences"] != sents:
                        raise Phase3FormatError(f"doc {did}: two different sentence lists ({cur['src']})")
                    cur["sentences"] = sents
                if labs is not None:
                    if cur["labels"] is not None and cur["labels"] != labs:
                        raise Phase3FormatError(f"doc {did}: two different label lists ({cur['src']})")
                    cur["labels"] = labs
    if not merged:
        raise Phase3FormatError(f"no documents parsed under {data_dir}")

    docs = sorted(merged.values(), key=lambda d: (d["split"], d["domain"], d["id"]))
    for d in docs:
        if d["sentences"] is None:
            raise Phase3FormatError(f"{d['split']}/{d['domain']}/{d['id']}: labels found but no sentences ({d['src']})")
        d["sentences"] = ["" if s is None else str(s) for s in d["sentences"]]
        if d["split"] == "test":
            if d["labels"] is not None:
                print(f"  WARNING: test doc {d['id']} carries labels - they are DROPPED and never used")
            d["labels"] = None
            continue
        if d["labels"] is None:
            raise Phase3FormatError(f"{d['split']}/{d['domain']}/{d['id']}: no labels for a {d['split']} document")
        if len(d["labels"]) != len(d["sentences"]):
            raise Phase3FormatError(f"{d['id']}: {len(d['sentences'])} sentences vs {len(d['labels'])} labels")
        bad = [l for l in d["labels"] if l not in LABEL_SET]
        if bad:
            raise Phase3FormatError(f"{d['id']}: unknown labels {sorted(set(map(str, bad)))[:5]}")

    have = {(d["split"], d["domain"]) for d in docs}
    if not any(s == "dev" for s, _ in have):
        print(f"  WARNING: no dev split found -> holding out {dev_docs_per_domain_fallback} train docs per domain")
        rng = random.Random(seed)
        for dom in DOMAINS:
            ids = sorted(d["id"] for d in docs if d["split"] == "train" and d["domain"] == dom)
            rng.shuffle(ids)
            hold = set(ids[:dev_docs_per_domain_fallback])
            for d in docs:
                if d["split"] == "train" and d["domain"] == dom and d["id"] in hold:
                    d["split"] = "dev"
    ids_by_split = defaultdict(list)
    for d in docs:
        ids_by_split[d["split"]].append(d["id"])
    for s, ids in ids_by_split.items():
        norm = [i[:-4] if i.endswith(".txt") else i for i in ids]
        dup = [i for i, c in Counter(norm).items() if c > 1]
        if dup:
            raise Phase3FormatError(f"{s}: document ids repeat across CL/IT (the scorer would reject this): {dup[:5]}")
    if verbose:
        print("  files used:", used_files)
        print("  label file:", label_path)
    return docs, label_path


def describe(docs):
    rows = []
    for split in ("train", "dev", "test"):
        for dom in DOMAINS:
            ds = [d for d in docs if d["split"] == split and d["domain"] == dom]
            n = sum(len(d["sentences"]) for d in ds)
            exp = EXPECTED_SENTENCES.get((split, dom))
            flag = "" if exp is None or exp == n else f"   <-- brief says {exp}"
            rows.append(f"  {split:<5} {dom}: {len(ds):>3} docs {n:>6} sentences{flag}")
    print("\n".join(rows))
    for split in ("train", "dev"):
        c = Counter(l for d in docs if d["split"] == split for l in d["labels"])
        print(f"  {split} label counts:", {l: c.get(l, 0) for l in LABELS})
        missing = [l for l in LABELS if c.get(l, 0) == 0]
        if missing:
            print(f"  NOTE: labels with zero {split} support: {missing}"
                  + ("  -> each scores F1=0 in official dev Macro-F1 whatever the model does" if split == "dev" else ""))

Writing /kaggle/working/rr_code/rr_data.py


In [3]:
%%writefile /kaggle/working/rr_code/rr_train.py
"""CS781 Phase 3 - Rhetorical Role prediction on top of the Phase-1 LegalLM.

One script, one experiment per call, one output directory per experiment:
    python rr_train.py --name e1_full --method full
    python rr_train.py --name e1_lorma --method lorma --hp '{"lr": 1e-3}'
    python rr_train.py --name smoke_full --method full --smoke        (2 docs/domain, a few steps)

Model ("packed-context hierarchical encoder"):
  consecutive sentences of a document are packed into one window (<= `window` tokens, an EOS between
  sentences) -> Phase-1 LegalLM with the causal mask removed (bidirectional=True), so every token sees
  the neighbouring sentences -> mean-pool each sentence's tokens -> + document-position bucket
  embedding + domain (CL/IT) embedding -> BiGRU over the sentences of the window -> 13-way softmax.
  Inference: overlapping windows (50% sentence overlap), centre-weighted average of probabilities.

Selection: official dev Macro-F1 from YOUR evaluate_phase3.py (run as a subprocess every epoch);
the in-script sklearn mirror must agree with it to 1e-6 or the run stops.
Test labels are never read. Submission: <out>/submission_<name>.zip with prediction.json at its root.
"""
import argparse
import hashlib
import json
import math
import os
import random
import re
import shutil
import subprocess
import sys
import time
import zipfile

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from rr_data import LABELS, DOMAINS, load_phase3, describe  # noqa: E402

L2I = {l: i for i, l in enumerate(LABELS)}
N_LABELS = len(LABELS)
DOM2I = {d: i for i, d in enumerate(DOMAINS)}

DEFAULTS = dict(
    bidirectional=True,       # drop the causal mask (Phase 2: bidirectional beat causal on this model)
    pooling="mean",           # "mean" | "last" (last is the natural choice when bidirectional=False)
    window=1024,              # tokens per packed window (Phase-1 max_seq_len = 1024)
    max_sent_tokens=256,      # a single sentence is truncated to this many tokens
    context="gru",            # "gru" | "none"   (BiGRU over the sentences of a window)
    gru_hidden=256,
    pos_buckets=16,           # document-position buckets (0 = off)
    domain_emb=True,
    epochs=12, patience=4, batch_size=4, eval_batch_size=8,
    lr=3e-5,                  # encoder (or adapter) learning rate
    head_lr=1e-3,             # pooling extras, GRU, classifier
    weight_decay=0.01, warmup=0.06, grad_clip=1.0, dropout=0.1,
    loss="ce",                # "ce" | "sqrt_weighted" (sqrt inverse-frequency, mean 1, clipped)
    label_smoothing=0.0,
    fp16=True, seed=42,
    max_steps_per_epoch=0,    # 0 = full epoch (smoke tests set this)
)
METHOD_DEFAULTS = {
    "full":   dict(lr=3e-5),
    "lora":   dict(lr=1e-3, rank=16, alpha=32, targets=["qkv", "proj", "w12", "w3"]),
    "rslora": dict(lr=1e-3, rank=16, alpha=32, targets=["qkv", "proj", "w12", "w3"]),
    "lorma":  dict(lr=1e-3, rank=16, alpha=32, targets=["qkv", "proj", "w12", "w3"]),
    "ia3":    dict(lr=3e-3),
    "frozen": dict(lr=0.0),   # linear-probe control: encoder frozen, only GRU/head train
}


def log(*a):
    print(*a, flush=True)


# ============================================================================ encoder (= Phase-1 weights)
class RMSNorm(nn.Module):
    def __init__(self, dim, eps):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        xf = x.float()
        return (xf * torch.rsqrt(xf.pow(2).mean(-1, keepdim=True) + self.eps) * self.weight).to(x.dtype)


def rope_tables(cfg):
    hd = cfg["d_model"] // cfg["n_head"]
    inv = 1.0 / (cfg.get("rope_theta", 10000.0) ** (torch.arange(0, hd, 2).float() / hd))
    f = torch.outer(torch.arange(cfg["max_seq_len"]).float(), inv)
    return f.cos(), f.sin()


def apply_rope(x, cos, sin):
    h = x.shape[-1] // 2
    x1, x2 = x[..., :h], x[..., h:]
    cos, sin = cos.to(x.dtype), sin.to(x.dtype)
    return torch.cat([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1)


class Attention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.h, self.hd = cfg["n_head"], cfg["d_model"] // cfg["n_head"]
        self.qkv = nn.Linear(cfg["d_model"], 3 * cfg["d_model"], bias=False)
        self.proj = nn.Linear(cfg["d_model"], cfg["d_model"], bias=False)

    def forward(self, x, cos, sin, mask, causal):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=2)
        q, k, v = (t.view(B, T, self.h, self.hd).transpose(1, 2) for t in (q, k, v))
        q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)
        if causal:
            y = F.scaled_dot_product_attention(q, k, v, is_causal=True)   # right padding never reaches real tokens
        else:
            y = F.scaled_dot_product_attention(q, k, v, attn_mask=mask)   # (B,1,1,T) True = attend
        return self.proj(y.transpose(1, 2).reshape(B, T, C))


class SwiGLU(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.w12 = nn.Linear(cfg["d_model"], 2 * cfg["d_ff"], bias=False)
        self.w3 = nn.Linear(cfg["d_ff"], cfg["d_model"], bias=False)

    def forward(self, x):
        a, b = self.w12(x).chunk(2, dim=-1)
        return self.w3(F.silu(a) * b)


class Block(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.norm1 = RMSNorm(cfg["d_model"], cfg.get("norm_eps", 1e-5))
        self.attn = Attention(cfg)
        self.norm2 = RMSNorm(cfg["d_model"], cfg.get("norm_eps", 1e-5))
        self.mlp = SwiGLU(cfg)

    def forward(self, x, cos, sin, mask, causal):
        x = x + self.attn(self.norm1(x), cos, sin, mask, causal)
        return x + self.mlp(self.norm2(x))


class LegalEncoder(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok = nn.Embedding(cfg["vocab_size"], cfg["d_model"])
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg["n_layer"])])
        self.norm = RMSNorm(cfg["d_model"], cfg.get("norm_eps", 1e-5))
        cos, sin = rope_tables(cfg)
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)

    def forward(self, ids, key_mask, causal):
        T = ids.shape[1]
        cos, sin = self.rope_cos[:T][None, None], self.rope_sin[:T][None, None]
        mask = None if causal else key_mask[:, None, None, :]
        x = self.tok(ids)
        for b in self.blocks:
            x = b(x, cos, sin, mask, causal)
        return self.norm(x)


# ============================================================================ PEFT modules
class LoRALinear(nn.Module):
    """W0 x + s * B A x ; s = alpha/r (LoRA) or alpha/sqrt(r) (rsLoRA). B = 0 at init."""
    def __init__(self, base, r, alpha, rs=False):
        super().__init__()
        self.base, self.s = base, (alpha / math.sqrt(r) if rs else alpha / r)
        self.A = nn.Parameter(torch.empty(r, base.in_features))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))

    def forward(self, x):
        return self.base(x) + self.s * F.linear(F.linear(x, self.A), self.B)


class LoRMALinear(nn.Module):
    """LoRMA+: (I + s B A) W0 x, computed as y0 + s B (A y0). B = 0 at init (same as your Phase 2 code)."""
    def __init__(self, base, r, alpha):
        super().__init__()
        self.base, self.s = base, alpha / r
        self.A = nn.Parameter(torch.empty(r, base.out_features))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))

    def forward(self, x):
        y0 = self.base(x)
        return y0 + self.s * F.linear(F.linear(y0, self.A), self.B)


class IA3QKV(nn.Module):
    """(IA)^3 on attention: rescale keys and values (queries untouched). Vectors start at 1."""
    def __init__(self, base, d):
        super().__init__()
        self.base = base
        self.l_k = nn.Parameter(torch.ones(d))
        self.l_v = nn.Parameter(torch.ones(d))
        self.register_buffer("ones_q", torch.ones(d), persistent=False)

    def forward(self, x):
        return self.base(x) * torch.cat([self.ones_q, self.l_k, self.l_v]).to(x.dtype)


class IA3In(nn.Module):
    """(IA)^3 on the FFN: rescale the d_ff activations entering w3."""
    def __init__(self, base):
        super().__init__()
        self.base = base
        self.l_ff = nn.Parameter(torch.ones(base.in_features))

    def forward(self, x):
        return self.base(x * self.l_ff.to(x.dtype))


def attach_peft(encoder, method, hp, d_model):
    n = 0
    for blk in encoder.blocks:
        if method == "ia3":
            blk.attn.qkv = IA3QKV(blk.attn.qkv, d_model)
            blk.mlp.w3 = IA3In(blk.mlp.w3)
            n += 2
            continue
        for parent, name in [(blk.attn, "qkv"), (blk.attn, "proj"), (blk.mlp, "w12"), (blk.mlp, "w3")]:
            if name not in hp["targets"]:
                continue
            base = getattr(parent, name)
            if method == "lora":
                mod = LoRALinear(base, hp["rank"], hp["alpha"])
            elif method == "rslora":
                mod = LoRALinear(base, hp["rank"], hp["alpha"], rs=True)
            elif method == "lorma":
                mod = LoRMALinear(base, hp["rank"], hp["alpha"])
            else:
                raise ValueError(method)
            setattr(parent, name, mod)
            n += 1
    return n


# ============================================================================ RR model
class RRModel(nn.Module):
    def __init__(self, cfg, hp):
        super().__init__()
        d = cfg["d_model"]
        self.hp = hp
        self.encoder = LegalEncoder(cfg)
        self.causal = not hp["bidirectional"]
        self.drop = nn.Dropout(hp["dropout"])
        self.pos_emb = nn.Embedding(hp["pos_buckets"], d) if hp["pos_buckets"] else None
        self.dom_emb = nn.Embedding(len(DOMAINS), d) if hp["domain_emb"] else None
        for e in (self.pos_emb, self.dom_emb):
            if e is not None:
                nn.init.normal_(e.weight, std=0.02)
        if hp["context"] == "gru":
            self.ctx = nn.GRU(d, hp["gru_hidden"], batch_first=True, bidirectional=True)
            out = d + 2 * hp["gru_hidden"]
        elif hp["context"] == "none":
            self.ctx, out = None, d
        else:
            raise ValueError(hp["context"])
        self.head = nn.Linear(out, N_LABELS)
        nn.init.normal_(self.head.weight, std=0.02)
        nn.init.zeros_(self.head.bias)

    def new_module_names(self):
        return ("pos_emb", "dom_emb", "ctx", "head")

    def forward(self, ids, key_mask, slot, n_sent, pos, dom):
        """ids/key_mask/slot: (B,T); slot = sentence index of each token, -1 for EOS/pad.
        n_sent: (B,) sentences per window; pos: (B,S) position bucket; dom: (B,)  ->  logits (B,S,13)"""
        h = self.encoder(ids, key_mask, self.causal).float()
        B, T, D = h.shape
        S = int(n_sent.max())
        valid = (slot >= 0)
        idx = slot.clamp(min=0)
        if self.hp["pooling"] == "mean":
            w = valid.float()
            sums = torch.zeros(B, S, D, device=h.device).scatter_add_(1, idx[..., None].expand(-1, -1, D), h * w[..., None])
            cnt = torch.zeros(B, S, device=h.device).scatter_add_(1, idx, w)
            v = sums / cnt.clamp(min=1.0)[..., None]
        elif self.hp["pooling"] == "last":
            tpos = torch.arange(T, device=h.device)[None].expand(B, T)
            lastpos = torch.full((B, S), -1, dtype=torch.long, device=h.device)
            lastpos.scatter_reduce_(1, idx, torch.where(valid, tpos, torch.full_like(tpos, -1)), reduce="amax")
            v = h.gather(1, lastpos.clamp(min=0)[..., None].expand(-1, -1, D))
        else:
            raise ValueError(self.hp["pooling"])
        if self.pos_emb is not None:
            v = v + self.pos_emb(pos)
        if self.dom_emb is not None:
            v = v + self.dom_emb(dom)[:, None]
        v = self.drop(v)
        if self.ctx is not None:
            packed = nn.utils.rnn.pack_padded_sequence(v, n_sent.cpu(), batch_first=True, enforce_sorted=False)
            c, _ = self.ctx(packed)
            c, _ = nn.utils.rnn.pad_packed_sequence(c, batch_first=True, total_length=S)
            v = torch.cat([v, self.drop(c)], -1)
        return self.head(v)


def load_phase1(model, model_dir):
    sd = torch.load(os.path.join(model_dir, "pytorch_model.bin"), map_location="cpu")
    if isinstance(sd, dict) and "model" in sd and isinstance(sd["model"], dict):
        sd = sd["model"]                                          # a training checkpoint instead of final weights
    sd = {k[len("_orig_mod."):] if k.startswith("_orig_mod.") else k: v for k, v in sd.items()}
    new = {"encoder." + k: v for k, v in sd.items() if not k.startswith("head.")}   # LM head is tied to tok
    missing, unexpected = model.load_state_dict(new, strict=False)
    new_mods = model.new_module_names()
    missing = [m for m in missing if not m.split(".")[0] in new_mods]
    if missing or unexpected:
        raise RuntimeError(f"Phase-1 weights do not match: missing={missing[:8]} unexpected={unexpected[:8]}")
    tied = torch.equal(sd["head.weight"], sd["tok.weight"]) if "head.weight" in sd else None
    return len(new), tied


def build_model(cfg, hp, method, model_dir):
    model = RRModel(cfg, hp)
    n_loaded, tied = load_phase1(model, model_dir)
    if method != "full":
        for p in model.encoder.parameters():
            p.requires_grad = False
    n_wrapped = attach_peft(model.encoder, method, hp, cfg["d_model"]) if method not in ("full", "frozen") else 0
    return model, n_loaded, tied, n_wrapped


def trainable_state(model):
    names = {n for n, p in model.named_parameters() if p.requires_grad}
    return {k: v.detach().cpu().clone() for k, v in model.state_dict().items() if k in names}


def load_trainable_state(model, state):
    names = {n for n, p in model.named_parameters() if p.requires_grad}
    if set(state) != names:
        raise RuntimeError(f"checkpoint/trainable mismatch: {sorted(set(state) ^ names)[:8]}")
    model.load_state_dict(state, strict=False)


# ============================================================================ data -> windows
class EncDoc:
    pass


def encode_docs(docs, tok, hp, eos_id):
    out = []
    for d in docs:
        e = EncDoc()
        e.id, e.domain, e.split = d["id"], d["domain"], d["split"]
        e.sent_ids = []
        for s in d["sentences"]:
            ids = tok.encode(s).ids[:hp["max_sent_tokens"]] if s.strip() else []
            e.sent_ids.append(ids if ids else [eos_id])          # empty sentence -> one placeholder token
        e.n = len(e.sent_ids)
        e.labels = np.array([L2I[l] for l in d["labels"]], dtype=np.int64) if d["labels"] is not None else None
        nb = hp["pos_buckets"] or 1
        e.pos = np.minimum((np.arange(e.n) * nb) // max(e.n, 1), nb - 1).astype(np.int64)
        out.append(e)
    return out


def pack_from(doc, start, budget):
    """Largest run of sentences starting at `start` that fits `budget` tokens (EOS after each sentence)."""
    used, end = 0, start
    while end < doc.n:
        need = len(doc.sent_ids[end]) + 1
        if end > start and used + need > budget:
            break
        used += need
        end += 1
    return end


def train_windows(docs, budget, rng):
    """Non-overlapping windows; random start offset per document and epoch -> varied context each epoch."""
    items = []
    for d in docs:
        first_end = pack_from(d, 0, budget)
        s = rng.randrange(0, max(1, first_end))
        if s > 0:
            items.append((d, 0, s))
        while s < d.n:
            e = pack_from(d, s, budget)
            items.append((d, s, e))
            s = e
    return items


def infer_windows(docs, budget):
    """Windows advance by half their sentence count -> every sentence is inside >= 1 window, mostly 2."""
    items = []
    for d in docs:
        s = 0
        while True:
            e = pack_from(d, s, budget)
            items.append((d, s, e))
            if e >= d.n:
                break
            s = max(s + 1, s + (e - s) // 2)
    return items


def collate(items, eos_id, with_labels=True):
    seqs, slots, ns = [], [], []
    for d, s, e in items:
        ids, sl = [], []
        for j, k in enumerate(range(s, e)):
            ids += d.sent_ids[k] + [eos_id]
            sl += [j] * len(d.sent_ids[k]) + [-1]
        seqs.append(ids); slots.append(sl); ns.append(e - s)
    B, T, S = len(items), max(map(len, seqs)), max(ns)
    ids = torch.full((B, T), eos_id, dtype=torch.long)
    km = torch.zeros((B, T), dtype=torch.bool)
    slot = torch.full((B, T), -1, dtype=torch.long)
    pos = torch.zeros((B, S), dtype=torch.long)
    lab = torch.full((B, S), -100, dtype=torch.long)
    dom = torch.tensor([DOM2I[d.domain] for d, _, _ in items], dtype=torch.long)
    for i, ((d, s, e), q, sl) in enumerate(zip(items, seqs, slots)):
        ids[i, :len(q)] = torch.tensor(q)
        km[i, :len(q)] = True
        slot[i, :len(sl)] = torch.tensor(sl)
        pos[i, :e - s] = torch.from_numpy(d.pos[s:e])
        if with_labels and d.labels is not None:
            lab[i, :e - s] = torch.from_numpy(d.labels[s:e])
    return ids, km, slot, torch.tensor(ns), pos, dom, lab


# ============================================================================ metrics + official evaluator
def sk_metrics(y_true, y_pred):
    from sklearn.metrics import f1_score, precision_score, recall_score, precision_recall_fscore_support, confusion_matrix
    m = dict(macro_f1=f1_score(y_true, y_pred, labels=LABELS, average="macro", zero_division=0),
             macro_p=precision_score(y_true, y_pred, labels=LABELS, average="macro", zero_division=0),
             macro_r=recall_score(y_true, y_pred, labels=LABELS, average="macro", zero_division=0))
    p, r, f, s = precision_recall_fscore_support(y_true, y_pred, labels=LABELS, zero_division=0)
    m["per_class"] = {l: dict(p=float(p[i]), r=float(r[i]), f1=float(f[i]), support=int(s[i])) for i, l in enumerate(LABELS)}
    m["confusion"] = confusion_matrix(y_true, y_pred, labels=LABELS).tolist()
    return m


def flatten(truth, pred, ids):
    yt, yp = [], []
    for d in ids:
        yt += truth[d]; yp += pred[d]
    return yt, yp


def full_report(dev_docs, preds):
    truth = {d.id: [LABELS[i] for i in d.labels] for d in dev_docs}
    rep = {"combined": sk_metrics(*flatten(truth, preds, list(truth)))}
    for dom in DOMAINS:
        ids = [d.id for d in dev_docs if d.domain == dom]
        if ids:
            rep[dom] = sk_metrics(*flatten(truth, preds, ids))
    return rep


class OfficialEvaluator:
    def __init__(self, evaluator_path, work_dir, dev_docs):
        self.path, self.dir = evaluator_path, work_dir
        os.makedirs(work_dir, exist_ok=True)
        self.labels_path = os.path.join(work_dir, "labels.json")
        json.dump(LABELS, open(self.labels_path, "w"))
        self.truth = {}
        for dom in DOMAINS:
            p = os.path.join(work_dir, f"dev_truth_{dom.lower()}.json")
            json.dump({d.id: [LABELS[i] for i in d.labels] for d in dev_docs if d.domain == dom},
                      open(p, "w"), ensure_ascii=False)
            self.truth[dom] = p

    def __call__(self, preds, tag):
        pp = os.path.join(self.dir, f"dev_prediction_{tag}.json")
        json.dump(preds, open(pp, "w"), ensure_ascii=False)
        cmd = [sys.executable, self.path, "--truth-cl", self.truth["CL"], "--truth-it", self.truth["IT"],
               "--prediction", pp, "--labels", self.labels_path]
        r = subprocess.run(cmd, capture_output=True, text=True)
        if r.returncode != 0:
            raise RuntimeError(f"official evaluator failed:\n{r.stdout}\n{r.stderr}")
        vals = {k: float(v) for k, v in re.findall(r"Macro-(F1|Precision|Recall):\s*([0-9.]+)", r.stdout)}
        if "F1" not in vals:
            raise RuntimeError(f"could not parse evaluator output:\n{r.stdout}")
        return vals, r.stdout


# ============================================================================ prediction
@torch.no_grad()
def predict(model, docs, hp, device, eos_id):
    model.eval()
    use_amp = hp["fp16"] and device.type == "cuda"
    items = infer_windows(docs, hp["window"])
    acc = {d.id: np.zeros((d.n, N_LABELS), dtype=np.float64) for d in docs}
    wsum = {d.id: np.zeros(d.n, dtype=np.float64) for d in docs}
    for k in range(0, len(items), hp["eval_batch_size"]):
        batch = items[k:k + hp["eval_batch_size"]]
        ids, km, slot, ns, pos, dom, _ = collate(batch, eos_id, with_labels=False)
        with torch.autocast("cuda", dtype=torch.float16, enabled=use_amp):
            logits = model(ids.to(device), km.to(device), slot.to(device), ns.to(device), pos.to(device), dom.to(device))
        probs = torch.softmax(logits.float(), -1).cpu().numpy()
        for i, (d, s, e) in enumerate(batch):
            j = np.arange(e - s)
            left = j if s > 0 else np.full(e - s, 10 ** 6)
            right = (e - s - 1) - j if e < d.n else np.full(e - s, 10 ** 6)
            w = np.minimum(np.minimum(left, right), 10 ** 3) + 1.0
            acc[d.id][s:e] += probs[i, :e - s] * w[:, None]
            wsum[d.id][s:e] += w
    preds, probs_out = {}, {}
    for d in docs:
        assert (wsum[d.id] > 0).all(), f"{d.id}: a sentence was not covered by any window"
        p = acc[d.id] / wsum[d.id][:, None]
        probs_out[d.id] = p
        preds[d.id] = [LABELS[i] for i in p.argmax(1)]
        assert len(preds[d.id]) == d.n
    return preds, probs_out


def dev_nll(dev_docs, probs):
    tot, n = 0.0, 0
    for d in dev_docs:
        p = probs[d.id][np.arange(d.n), d.labels]
        tot += -np.log(np.clip(p, 1e-12, 1)).sum(); n += d.n
    return tot / max(n, 1)


def class_weights(train_docs, hp):
    if hp["loss"] == "ce":
        return None
    if hp["loss"] != "sqrt_weighted":
        raise ValueError(hp["loss"])
    c = np.bincount(np.concatenate([d.labels for d in train_docs]), minlength=N_LABELS).astype(float)
    w = np.where(c > 0, np.sqrt(c.sum() / np.maximum(c, 1)), 1.0)
    w = w / (w * c).sum() * c.sum()                     # sentence-weighted mean = 1
    return torch.tensor(np.clip(w, 0.25, 10.0), dtype=torch.float32)


# ============================================================================ main
def param_fingerprint(params):
    h = hashlib.sha1()
    for p in params:
        h.update(p.detach().float().cpu().numpy().tobytes()[:4096])
    return h.hexdigest()


def write_submission(preds, test_docs, out_dir, name):
    clean = {}
    for d in test_docs:
        lab = preds[d.id]
        assert len(lab) == d.n and all(l in L2I for l in lab), d.id
        clean[d.id] = lab
    pj = os.path.join(out_dir, "prediction.json")
    json.dump(clean, open(pj, "w"), ensure_ascii=False)
    zp = os.path.join(out_dir, f"submission_{name}.zip")
    with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(pj, arcname="prediction.json")
    with zipfile.ZipFile(zp) as z:
        assert z.namelist() == ["prediction.json"]
    return zp, len(clean), sum(len(v) for v in clean.values())


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--name", required=True)
    ap.add_argument("--method", required=True, choices=sorted(METHOD_DEFAULTS))
    ap.add_argument("--hp", default="{}", help="JSON dict of overrides")
    ap.add_argument("--paths", default="/kaggle/working/rr_paths.json",
                    help="JSON with data_dir, model_dir, tokenizer, evaluator, out_root")
    ap.add_argument("--smoke", action="store_true")
    ap.add_argument("--force", action="store_true", help="re-run even if the experiment finished before")
    args = ap.parse_args()

    paths = json.load(open(args.paths))
    hp = {**DEFAULTS, **METHOD_DEFAULTS[args.method], **json.loads(args.hp)}
    unknown = set(json.loads(args.hp)) - set(DEFAULTS) - set(METHOD_DEFAULTS[args.method])
    if unknown:
        raise SystemExit(f"unknown hp keys: {sorted(unknown)}")
    name = ("_smoke_" + args.name) if args.smoke else args.name
    out_dir = os.path.join(paths["out_root"], name)
    if args.smoke and os.path.isdir(out_dir):
        shutil.rmtree(out_dir)
    if os.path.exists(os.path.join(out_dir, "DONE")) and not args.force:
        log(f"{name} already finished ({out_dir}); use a new --name, or --force to redo it")
        return
    os.makedirs(out_dir, exist_ok=True)

    random.seed(hp["seed"]); np.random.seed(hp["seed"]); torch.manual_seed(hp["seed"])
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if device.type == "cuda":
        log(f"[{name}] GPU {torch.cuda.get_device_name(0)} | visible GPUs {torch.cuda.device_count()} | "
            f"CUDA_VISIBLE_DEVICES={os.environ.get('CUDA_VISIBLE_DEVICES')}")
    else:
        log(f"[{name}] running on CPU")

    # ---- data
    from tokenizers import Tokenizer
    tok = Tokenizer.from_file(paths["tokenizer"])
    eos_id = tok.token_to_id("<|endoftext|>")
    assert eos_id is not None, "tokenizer has no <|endoftext|>"
    docs, _ = load_phase3(paths["data_dir"], seed=hp["seed"], verbose=False)
    if args.smoke:
        keep = []
        for split in ("train", "dev", "test"):
            for dom in DOMAINS:
                keep += [d for d in docs if d["split"] == split and d["domain"] == dom][:2]
        docs = keep
        hp.update(epochs=2, patience=2, max_steps_per_epoch=6)
    describe(docs)
    enc = encode_docs(docs, tok, hp, eos_id)
    train_docs = [d for d in enc if d.split == "train"]
    dev_docs = [d for d in enc if d.split == "dev"]
    test_docs = [d for d in enc if d.split == "test"]
    tl = [len(s) for d in train_docs for s in d.sent_ids]
    log(f"sentence tokens (train): median {int(np.median(tl))}, p95 {int(np.percentile(tl, 95))}, max {max(tl)} "
        f"(cap {hp['max_sent_tokens']}); truncated {sum(t >= hp['max_sent_tokens'] for t in tl)}")

    # ---- model
    cfg = json.load(open(os.path.join(paths["model_dir"], "config.json")))
    model, n_loaded, tied, n_wrapped = build_model(cfg, hp, args.method, paths["model_dir"])
    model.to(device)
    tot = sum(p.numel() for p in model.parameters())
    trn = sum(p.numel() for p in model.parameters() if p.requires_grad)
    log(f"Phase-1 tensors loaded {n_loaded} (strict, LM head tied={tied}) | PEFT modules {n_wrapped} | "
        f"params {tot / 1e6:.2f}M, trainable {trn / 1e6:.3f}M ({100 * trn / tot:.2f}%)")
    frozen = [p for n, p in model.named_parameters() if not p.requires_grad]
    frozen_fp0 = param_fingerprint(frozen[:6]) if frozen else None
    json.dump({"name": name, "method": args.method, "hp": hp, "model_cfg": cfg, "paths": paths,
               "params_total": tot, "params_trainable": trn}, open(os.path.join(out_dir, "run_config.json"), "w"), indent=1)

    # ---- optimiser
    new_mods = model.new_module_names()
    enc_dec, enc_nodec, head = [], [], []
    for n, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if n.split(".")[0] in new_mods:
            head.append(p)
        elif p.ndim >= 2:
            enc_dec.append(p)
        else:
            enc_nodec.append(p)
    groups = [g for g in [{"params": enc_dec, "lr": hp["lr"], "weight_decay": hp["weight_decay"]},
                          {"params": enc_nodec, "lr": hp["lr"], "weight_decay": 0.0},
                          {"params": head, "lr": hp["head_lr"], "weight_decay": hp["weight_decay"]}] if g["params"]]
    opt = torch.optim.AdamW(groups)
    n0 = len(train_windows(train_docs, hp["window"], random.Random(0)))
    spe = math.ceil(n0 / hp["batch_size"])
    if hp["max_steps_per_epoch"]:
        spe = min(spe, hp["max_steps_per_epoch"])
    total = spe * hp["epochs"]
    warm = max(1, int(hp["warmup"] * total))
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: min(1.0, (s + 1) / warm) * max(0.0, min(1.0, (total - s) / max(1, total - warm))))
    use_amp = hp["fp16"] and device.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    cw = class_weights(train_docs, hp)
    cw = cw.to(device) if cw is not None else None
    log(f"windows/epoch ~{n0} | steps/epoch {spe} | total {total} | warmup {warm} | class weights "
        f"{None if cw is None else [round(x, 2) for x in cw.tolist()]}")

    ev = OfficialEvaluator(paths["evaluator"], os.path.join(out_dir, "eval"), dev_docs)

    # ---- resume
    start_ep, history, best, bad = 1, [], None, 0
    last_p, best_p = os.path.join(out_dir, "last.pt"), os.path.join(out_dir, "best.pt")
    if os.path.exists(last_p):
        ck = torch.load(last_p, map_location="cpu", weights_only=False)
        load_trainable_state(model, ck["model"])
        opt.load_state_dict(ck["opt"]); sched.load_state_dict(ck["sched"]); scaler.load_state_dict(ck["scaler"])
        start_ep, history, best, bad = ck["epoch"] + 1, ck["history"], ck["best"], ck["bad"]
        random.setstate(ck["py_rng"]); np.random.set_state(ck["np_rng"]); torch.set_rng_state(ck["torch_rng"])
        log(f"RESUMED after epoch {ck['epoch']} (best so far {best and best['dev_f1_official']})")
        del ck

    rng = random.Random(hp["seed"] * 1000 + start_ep)
    trainable = [p for p in model.parameters() if p.requires_grad]
    for ep in range(start_ep, hp["epochs"] + 1):
        if bad >= hp["patience"]:
            break
        model.train()
        items = train_windows(train_docs, hp["window"], rng)
        rng.shuffle(items)
        t0, tot_loss, nb, nonfinite, gnorms = time.time(), 0.0, 0, 0, []
        for k in range(0, len(items), hp["batch_size"]):
            if hp["max_steps_per_epoch"] and nb >= hp["max_steps_per_epoch"]:
                break
            ids, km, slot, ns, pos, dom, lab = collate(items[k:k + hp["batch_size"]], eos_id)
            ids, km, slot, pos, dom, lab = (t.to(device) for t in (ids, km, slot, pos, dom, lab))
            with torch.autocast("cuda", dtype=torch.float16, enabled=use_amp):
                logits = model(ids, km, slot, ns.to(device), pos, dom)
            logits = logits.float()
            valid = lab != -100
            loss = F.cross_entropy(logits[valid], lab[valid], weight=cw, label_smoothing=hp["label_smoothing"])
            opt.zero_grad(set_to_none=True)
            if not torch.isfinite(loss):
                nonfinite += 1
                if nonfinite > 20:
                    raise RuntimeError("more than 20 non-finite losses in one epoch - stopping")
                continue
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            gn = torch.nn.utils.clip_grad_norm_(trainable, hp["grad_clip"])
            scale_before = scaler.get_scale()
            scaler.step(opt); scaler.update()
            if scaler.get_scale() >= scale_before:              # optimizer actually stepped
                sched.step()
            if torch.isfinite(gn):
                gnorms.append(float(gn))
            tot_loss += loss.item(); nb += 1
            if ep == start_ep and nb == 1:
                no_grad = [n for n, p in model.named_parameters() if p.requires_grad and p.grad is None]
                log(f"  first step: loss {loss.item():.4f} | grad-norm {float(gn):.3f} | trainable params without grad: "
                    f"{len(no_grad)} {no_grad[:3]}")
        train_s = time.time() - t0
        if nb == 0:
            raise RuntimeError("no optimisation step was taken this epoch")

        t1 = time.time()
        preds, probs = predict(model, dev_docs, hp, device, eos_id)
        off, _ = ev(preds, f"ep{ep}")
        rep = full_report(dev_docs, preds)
        mine = rep["combined"]["macro_f1"]
        if abs(mine - off["F1"]) > 1e-5:
            raise RuntimeError(f"in-script Macro-F1 {mine:.6f} != official {off['F1']:.6f}")
        rec = {"epoch": ep, "train_loss": tot_loss / nb, "dev_nll": dev_nll(dev_docs, probs),
               "dev_f1_official": off["F1"], "dev_p": off["Precision"], "dev_r": off["Recall"],
               **{f"dev_f1_{dm}": rep[dm]["macro_f1"] for dm in DOMAINS if dm in rep},
               "grad_norm_mean": float(np.mean(gnorms)) if gnorms else None, "nonfinite_steps": nonfinite,
               "steps": nb, "train_s": train_s, "eval_s": time.time() - t1,
               "lr_now": [g["lr"] for g in opt.param_groups]}
        improved = best is None or off["F1"] > best["dev_f1_official"]
        if improved:
            best, bad = dict(rec), 0
            torch.save(trainable_state(model), best_p)
            json.dump(preds, open(os.path.join(out_dir, "dev_predictions_best.json"), "w"), ensure_ascii=False)
            json.dump(rep, open(os.path.join(out_dir, "dev_report_best.json"), "w"), indent=1)
        else:
            bad += 1
        history.append(rec)
        log(f"  ep {ep:>2}/{hp['epochs']} loss {rec['train_loss']:.4f} dev-nll {rec['dev_nll']:.4f} | OFFICIAL dev "
            f"Macro-F1 {off['F1']:.4f} (P {off['Precision']:.3f} R {off['Recall']:.3f}) | CL {rec.get('dev_f1_CL', 0):.4f} "
            f"IT {rec.get('dev_f1_IT', 0):.4f}{'  *' if improved else ''}  [{train_s:.0f}s+{rec['eval_s']:.0f}s]")
        if ep == start_ep and frozen_fp0 is not None:
            assert param_fingerprint(frozen[:6]) == frozen_fp0, "frozen pretrained weights changed!"
            if args.method in ("lora", "rslora", "lorma"):
                bn = sum(float(m.B.norm()) for m in model.modules() if isinstance(m, (LoRALinear, LoRMALinear)))
                assert bn > 0, "LoRA/LoRMA B matrices are still zero - adapters did not update"
                log(f"  check: frozen weights unchanged, adapter |B| sum = {bn:.4f}")
            elif args.method == "ia3":
                dv = sum(float((m.l_k - 1).abs().sum() + (m.l_v - 1).abs().sum()) for m in model.modules() if isinstance(m, IA3QKV))
                assert dv > 0, "IA3 vectors did not move"
                log(f"  check: frozen weights unchanged, IA3 |l-1| sum = {dv:.4f}")
        torch.save({"model": trainable_state(model), "opt": opt.state_dict(), "sched": sched.state_dict(),
                    "scaler": scaler.state_dict(), "epoch": ep, "history": history, "best": best, "bad": bad,
                    "py_rng": random.getstate(), "np_rng": np.random.get_state(), "torch_rng": torch.get_rng_state()},
                   last_p + ".tmp")
        os.replace(last_p + ".tmp", last_p)
        if bad >= hp["patience"]:
            log(f"  early stop: no official dev improvement for {hp['patience']} epochs")

    # ---- best checkpoint -> dev report + test submission
    load_trainable_state(model, torch.load(best_p, map_location="cpu"))
    model.to(device)
    preds, _ = predict(model, dev_docs, hp, device, eos_id)
    off, off_txt = ev(preds, "best")
    if abs(off["F1"] - best["dev_f1_official"]) > 1e-6:      # fp16 kernels can flip a near-tie
        log(f"  NOTE: reloaded best checkpoint scores {off['F1']:.6f} vs {best['dev_f1_official']:.6f} during training")
    rep = json.load(open(os.path.join(out_dir, "dev_report_best.json")))
    t0 = time.time()
    test_preds, _ = predict(model, test_docs, hp, device, eos_id)
    zp, n_docs, n_sent = write_submission(test_preds, test_docs, out_dir, name)
    if n_docs != 10 and not args.smoke:
        log(f"  WARNING: submission has {n_docs} test documents (brief expects 10)")

    log("\n" + "=" * 90 + f"\n{name} | best epoch {best['epoch']} | OFFICIAL dev Macro-F1 {off['F1']:.4f}  "
        f"(P {off['Precision']:.4f} R {off['Recall']:.4f})")
    log("official evaluator output:\n" + off_txt.strip())
    for dm in ["combined"] + [d for d in DOMAINS if d in rep]:
        log(f"  {dm:<8} macro-F1 {rep[dm]['macro_f1']:.4f}  P {rep[dm]['macro_p']:.4f}  R {rep[dm]['macro_r']:.4f}")
    log(f"  {'label':<24}{'P':>7}{'R':>7}{'F1':>7}{'supp':>6} | CL F1  IT F1")
    for l in LABELS:
        c = rep["combined"]["per_class"][l]
        dom_f = "  ".join(f"{rep[d]['per_class'][l]['f1']:.3f}" if d in rep else "  -  " for d in DOMAINS)
        log(f"  {l:<24}{c['p']:>7.3f}{c['r']:>7.3f}{c['f1']:>7.3f}{c['support']:>6} | {dom_f}")
    log(f"  confusion (rows = gold, cols = predicted, order = {LABELS}):")
    for l, row in zip(LABELS, rep["combined"]["confusion"]):
        log(f"  {l[:12]:<12} " + " ".join(f"{x:>4}" for x in row))
    log(f"  submission: {zp}  ({n_docs} docs, {n_sent} sentences, prediction.json at zip root)")

    metrics = {"name": name, "method": args.method, "best_epoch": best["epoch"],
               "dev_f1_official": off["F1"], "dev_p_official": off["Precision"], "dev_r_official": off["Recall"],
               **{f"dev_f1_{d}": rep[d]["macro_f1"] for d in DOMAINS if d in rep},
               "per_class_f1": {l: rep["combined"]["per_class"][l]["f1"] for l in LABELS},
               "params_total": tot, "params_trainable": trn, "history": history, "hp": hp,
               "test_docs": n_docs, "test_sentences": n_sent, "test_infer_s": time.time() - t0,
               "peak_gpu_gb": torch.cuda.max_memory_allocated() / 2 ** 30 if device.type == "cuda" else None}
    json.dump(metrics, open(os.path.join(out_dir, "metrics.json"), "w"), indent=1)
    open(os.path.join(out_dir, "DONE"), "w").write(time.ctime())
    if args.smoke:
        log("SMOKE TEST PASSED")


if __name__ == "__main__":
    main()

Writing /kaggle/working/rr_code/rr_train.py


## 3. Inspect the data - check this output before training

The loader detects the file layout. If this cell raises an error, or the sentence counts differ from the brief, paste the whole output back to me; do not train on a wrong parse.

In [4]:
sys.path.insert(0, CODE_DIR)
import importlib, rr_data; importlib.reload(rr_data)
print(rr_data.inventory(DATA_DIR, max_preview=250))
print()
docs, label_file = rr_data.load_phase3(DATA_DIR)
rr_data.describe(docs)

import numpy as np
from tokenizers import Tokenizer
tok = Tokenizer.from_file(TOKENIZER)
print("EOS id:", tok.token_to_id("<|endoftext|>"), "| vocab:", tok.get_vocab_size())
lens = [len(tok.encode(s).ids) for d in docs if d["split"] == "train" for s in d["sentences"]]
print(f"train sentence tokens: median {np.median(lens):.0f}  p90 {np.percentile(lens, 90):.0f}  p99 {np.percentile(lens, 99):.0f}  max {max(lens)}")
for split in ("train", "dev", "test"):
    n = [len(d["sentences"]) for d in docs if d["split"] == split]
    print(f"{split}: sentences/doc min {min(n)} median {int(np.median(n))} max {max(n)}")
print("example test ids:", [d["id"] for d in docs if d["split"] == "test"][:10])

Test/CL_test.json  (274.0 KB)  split=test domain=CL
    preview: {\n  "HC_Raghupati_Singhania__vs__Competition_Commission_ofDE201616051616063761COM513489": [\n    "Manmohan, J. 1 .",\n    "Present writ petition has been filed, challenging the notice dated 26th February, 2016 and respondent Number 1s order dated 18th 
Test/IT_test.json  (189.8 KB)  split=test domain=IT
    preview: {\n  "SC_2009_710": [\n    "B. Sinha, J. Leave granted.",\n    "These two appeals, being interconnected, were taken up for hearing together and are being disposed of by this common judgment.",\n    "They arise out of a common judgment and order dated 2.3
Train/CL_dev.json  (328.4 KB)  split=None domain=CL
    preview: [\n  {\n    "id": "CCI_Noida_Software_Technology_Park_Ltd_vs_Star_India_PCO201808081816041524COM760948",\n    "text": [\n      "Noida Software Technology Park Limited (hereinafter, the Informant) filed the present information under Section 19(1)(a) of th
Train/CL_train.json  (2402.6 KB)  split=t

## 4. Smoke tests (about 5 minutes)

Two docs per domain, 2 epochs of 6 steps, real Phase-1 weights on a T4. They check strict weight loading, finite loss, gradients on every trainable tensor, frozen weights unchanged and adapters moving (LoRMA), that the official evaluator runs and agrees with the in-script metric, and that the submission zip is well formed. Output goes to `_smoke_*` folders, which are recreated each time.

In [5]:
def run(name, method, hp=None, gpu=0, smoke=False):
    cmd = [sys.executable, os.path.join(CODE_DIR, "rr_train.py"), "--name", name, "--method", method,
           "--hp", json.dumps(hp or {}), "--paths", "/kaggle/working/rr_paths.json"] + (["--smoke"] if smoke else [])
    env = {**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu), "PYTHONUNBUFFERED": "1"}
    r = subprocess.run(cmd, env=env, capture_output=True, text=True)
    print(r.stdout[-6000:]); print(r.stderr[-4000:])
    assert r.returncode == 0, f"{name} failed"

run("full", "full", smoke=True, gpu=0)
run("lorma", "lorma", smoke=True, gpu=1)

[_smoke_full] GPU Tesla T4 | visible GPUs 1 | CUDA_VISIBLE_DEVICES=0
  skip Train/CL_dev.json: cannot tell train/dev/test from its path
  skip Train/IT_dev.json: cannot tell train/dev/test from its path
  train CL:   2 docs     73 sentences   <-- brief says 10593
  train IT:   2 docs    171 sentences   <-- brief says 6252
  dev   CL:   2 docs    108 sentences   <-- brief says 1396
  dev   IT:   2 docs    508 sentences   <-- brief says 746
  test  CL:   2 docs    344 sentences   <-- brief says 1339
  test  IT:   2 docs    199 sentences   <-- brief says 858
  train label counts: {'Fact': 63, 'Issue': 3, 'ArgumentPetitioner': 23, 'ArgumentRespondent': 17, 'Statute': 39, 'Dissent': 0, 'PrecedentReliedUpon': 23, 'PrecedentNotReliedUpon': 4, 'PrecedentOverruled': 0, 'RulingByLowerCourt': 1, 'RatioOfTheDecision': 47, 'RulingByPresentCourt': 13, 'None': 11}
  NOTE: labels with zero train support: ['Dissent', 'PrecedentOverruled']
  dev label counts: {'Fact': 86, 'Issue': 10, 'ArgumentPetitione

## 5. Round 1 - context vs no context, both full fine-tuning, in parallel on the two T4s

Same model, optimiser, learning rates, epochs, seed and selection rule; only the input context differs. `e0` uses `window=1` (exactly one sentence per window), no BiGRU, no position or domain features.
Estimated wall time: about 30-45 min (an estimate, not a measurement). Progress prints every 2 minutes; full logs are in `/kaggle/working/rr/logs/`.
If the session dies, run the same cell again: each run resumes from its `last.pt`, and finished runs are skipped.

In [6]:
def run_parallel(jobs, poll=120):
    """jobs: list of (name, method, hp, gpu). One process per GPU, waits for all, streams the epoch lines."""
    os.makedirs(os.path.join(OUT_ROOT, "logs"), exist_ok=True)
    procs = []
    for name, method, hp, gpu in jobs:
        lp = os.path.join(OUT_ROOT, "logs", f"{name}.log")
        cmd = [sys.executable, os.path.join(CODE_DIR, "rr_train.py"), "--name", name, "--method", method,
               "--hp", json.dumps(hp), "--paths", "/kaggle/working/rr_paths.json"]
        env = {**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu), "PYTHONUNBUFFERED": "1"}
        procs.append((name, lp, subprocess.Popen(cmd, env=env, stdout=open(lp, "a"), stderr=subprocess.STDOUT)))
        print(f"started {name} on GPU {gpu} -> {lp}")
    seen = {n: 0 for n, _, _ in procs}
    while True:
        time.sleep(poll)
        for n, lp, p in procs:
            lines = open(lp).read().splitlines()
            for l in lines[seen[n]:]:
                if l.lstrip().startswith(("ep ", "first step", "check:", "early stop", "RESUMED", "Phase-1", "Trace", "Error", "RuntimeError")) or "already finished" in l:
                    print(f"[{n}] {l.strip()}")
            seen[n] = len(lines)
        if all(p.poll() is not None for _, _, p in procs):
            break
    for n, lp, p in procs:
        print(f"\n{'#' * 30} {n}: exit code {p.returncode}")
        txt = open(lp).read()
        i = txt.rfind("=" * 90)
        print(txt[i:] if i >= 0 else txt[-5000:])

ISOLATED = dict(window=1, context="none", pos_buckets=0, domain_emb=False, batch_size=32, eval_batch_size=128)
run_parallel([("e0_sent_full", "full", ISOLATED, 0),
              ("e1_ctx_full",  "full", {}, 1)])

started e0_sent_full on GPU 0 -> /kaggle/working/rr/logs/e0_sent_full.log
started e1_ctx_full on GPU 1 -> /kaggle/working/rr/logs/e1_ctx_full.log
[e0_sent_full] Phase-1 tensors loaded 74 (strict, LM head tied=True) | PEFT modules 0 | params 109.54M, trainable 109.540M (100.00%)
[e0_sent_full] first step: loss 3.1337 | grad-norm inf | trainable params without grad: 0 []
[e1_ctx_full] Phase-1 tensors loaded 74 (strict, LM head tied=True) | PEFT modules 0 | params 111.14M, trainable 111.136M (100.00%)
[e1_ctx_full] first step: loss 2.6444 | grad-norm inf | trainable params without grad: 0 []
[e1_ctx_full] ep  1/12 loss 1.9301 dev-nll 1.7399 | OFFICIAL dev Macro-F1 0.2412 (P 0.347 R 0.270) | CL 0.2381 IT 0.2092  *  [50s+6s]
[e0_sent_full] ep  1/12 loss 1.6124 dev-nll 1.5488 | OFFICIAL dev Macro-F1 0.3527 (P 0.439 R 0.325) | CL 0.3125 IT 0.3709  *  [123s+8s]
[e1_ctx_full] ep  2/12 loss 1.2272 dev-nll 1.7491 | OFFICIAL dev Macro-F1 0.2735 (P 0.402 R 0.279) | CL 0.2775 IT 0.2692  *  [54s+5s]


## 6. Compare finished runs (official dev Macro-F1, same dev split)

In [7]:
import numpy as np
LABELS = rr_data.LABELS
rows = [json.load(open(p)) for p in sorted(glob.glob(os.path.join(OUT_ROOT, "*", "metrics.json"))) if "/_smoke_" not in p]
hdr = f"{'run':<20}{'method':<8}{'ep':>4}{'devF1':>8}{'P':>7}{'R':>7}{'CL F1':>8}{'IT F1':>8}{'trainable':>11}{'GPU GB':>8}"
print(hdr); print("-" * len(hdr))
for m in rows:
    print(f"{m['name']:<20}{m['method']:<8}{m['best_epoch']:>4}{m['dev_f1_official']:>8.4f}{m['dev_p_official']:>7.3f}"
          f"{m['dev_r_official']:>7.3f}{m.get('dev_f1_CL', float('nan')):>8.4f}{m.get('dev_f1_IT', float('nan')):>8.4f}"
          f"{m['params_trainable'] / 1e6:>10.2f}M{(m['peak_gpu_gb'] or 0):>8.2f}")
print("\nper-class dev F1:")
print(f"{'label':<24}" + "".join(f"{m['name'][:14]:>15}" for m in rows))
for l in LABELS:
    print(f"{l:<24}" + "".join(f"{m['per_class_f1'][l]:>15.3f}" for m in rows))
print("\noverfitting view (train loss / dev NLL / official dev F1 per epoch):")
for m in rows:
    print(m["name"], " | ".join(f"ep{h['epoch']} {h['train_loss']:.3f}/{h['dev_nll']:.3f}/{h['dev_f1_official']:.3f}" for h in m["history"]))

run                 method    ep   devF1      P      R   CL F1   IT F1  trainable  GPU GB
-----------------------------------------------------------------------------------------
e0_sent_full        full       1  0.3527  0.439  0.325  0.3125  0.3709    109.54M    5.72
e1_ctx_full         full       7  0.3931  0.493  0.374  0.3637  0.3906    111.14M    3.80

per-class dev F1:
label                      e0_sent_full    e1_ctx_full
Fact                              0.627          0.721
Issue                             0.625          0.733
ArgumentPetitioner                0.415          0.499
ArgumentRespondent                0.184          0.178
Statute                           0.453          0.490
Dissent                           0.000          0.000
PrecedentReliedUpon               0.374          0.452
PrecedentNotReliedUpon            0.088          0.019
PrecedentOverruled                0.000          0.000
RulingByLowerCourt                0.234          0.338
RatioOfTheDecisi

## 7. Round 2 - run after reading round 1

Both runs use the `e1` pipeline. Run this cell only if `e1_ctx_full` beat `e0_sent_full` by a clear margin; with 10 dev documents a gap under ~0.01 is noise. Otherwise send me the cell 6 output first.

In [8]:
run_parallel([("e2_ctx_lorma",      "lorma", {}, 0),
              ("e2_ctx_full_sqrtw", "full",  {"loss": "sqrt_weighted"}, 1)])

started e2_ctx_lorma on GPU 0 -> /kaggle/working/rr/logs/e2_ctx_lorma.log
started e2_ctx_full_sqrtw on GPU 1 -> /kaggle/working/rr/logs/e2_ctx_full_sqrtw.log
[e2_ctx_lorma] Phase-1 tensors loaded 74 (strict, LM head tied=True) | PEFT modules 48 | params 114.18M, trainable 4.654M (4.08%)
[e2_ctx_lorma] first step: loss 2.6444 | grad-norm inf | trainable params without grad: 0 []
[e2_ctx_lorma] ep  1/12 loss 2.3562 dev-nll 2.6275 | OFFICIAL dev Macro-F1 0.0236 (P 0.014 R 0.077) | CL 0.0241 IT 0.0230  *  [51s+7s]
[e2_ctx_lorma] check: frozen weights unchanged, adapter |B| sum = 48.5132
[e2_ctx_lorma] Traceback (most recent call last):
[e2_ctx_lorma] RuntimeError: more than 20 non-finite losses in one epoch - stopping
[e2_ctx_full_sqrtw] Phase-1 tensors loaded 74 (strict, LM head tied=True) | PEFT modules 0 | params 111.14M, trainable 111.136M (100.00%)
[e2_ctx_full_sqrtw] first step: loss 2.5829 | grad-norm inf | trainable params without grad: 0 []
[e2_ctx_full_sqrtw] ep  1/12 loss 2.0984

## 8. Download submissions and reports

In [9]:
import shutil, zipfile
from IPython.display import FileLink, display
os.chdir("/kaggle/working")
done = sorted(p for p in glob.glob(os.path.join(OUT_ROOT, "*", "DONE")) if "/_smoke_" not in p)
with zipfile.ZipFile("rr_results.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for p in done:
        d = os.path.dirname(p); n = os.path.basename(d)
        sub = os.path.join(d, f"submission_{n}.zip")
        shutil.copy(sub, f"submission_{n}.zip"); display(FileLink(f"submission_{n}.zip"))
        for f in ("metrics.json", "dev_report_best.json", "run_config.json", "dev_predictions_best.json"):
            z.write(os.path.join(d, f), arcname=f"{n}/{f}")
        lp = os.path.join(OUT_ROOT, "logs", f"{n}.log")
        if os.path.exists(lp):
            z.write(lp, arcname=f"{n}/train.log")
display(FileLink("rr_results.zip"))

/kaggle/working/submission_e0_sent_full.zip

/kaggle/working/submission_e1_ctx_full.zip

/kaggle/working/submission_e2_ctx_full_sqrtw.zip

/kaggle/working/rr_results.zip

In [15]:
# ===================== CS781 Phase 3 - v2 setup: ONE plain-Python cell (no %% magics) =====================
# Paste this whole file into ONE new code cell and run it. It (1) overwrites rr_data.py / rr_train.py with the
# fixed v2 code, (2) re-checks the data, (3) defines run / run_parallel / compare / download helpers.
# It trains nothing. Needs /kaggle/working/rr_paths.json from your original paths cell (re-run that first
# if the kernel restarted).
import os, sys, json, glob, subprocess, time, importlib

PATHS = json.load(open("/kaggle/working/rr_paths.json"))
DATA_DIR, TOKENIZER, OUT_ROOT = PATHS["data_dir"], PATHS["tokenizer"], PATHS["out_root"]
CODE_DIR = "/kaggle/working/rr_code"
os.makedirs(CODE_DIR, exist_ok=True)

RR_DATA_SRC = r'''"""CS781 Phase 3 (Rhetorical Roles) - data discovery, format detection and validation.

The exact file layout of Phase_3_RR/ is detected at runtime, never assumed. Supported shapes
(JSON, JSONL, CSV/TSV), per split (train / dev / test) and domain (CL / IT):
  * {doc_id: {"sentences": [...], "labels": [...]}}             (any common key names)
  * [{"id": ..., "sentences": [...], "labels": [...]}, ...]
  * {doc_id: [sentence, ...]}  +  {doc_id: [label, ...]}        (text and truth in separate files)
  * {doc_id: [{"text"/"sentence": ..., "label": ...}, ...]}
  * legal-eval style [{"id", "annotations": [{"result": [{"value": {"text", "labels": [...]}}]}]}]
  * CSV/TSV rows with a document-id column, a sentence column and (optionally) a label column
Split and domain come from the file path (e.g. .../CL/train.json, it_dev.json) or a record field.
Anything unrecognised stops with an explicit message instead of guessing.
"""
import csv
import json
import os
import random
import re
from collections import Counter, defaultdict

LABELS = ["Fact", "Issue", "ArgumentPetitioner", "ArgumentRespondent", "Statute", "Dissent",
          "PrecedentReliedUpon", "PrecedentNotReliedUpon", "PrecedentOverruled", "RulingByLowerCourt",
          "RatioOfTheDecision", "RulingByPresentCourt", "None"]
LABEL_SET = set(LABELS)
DOMAINS = ["CL", "IT"]
EXPECTED_SENTENCES = {("train", "CL"): 10593, ("train", "IT"): 6252, ("dev", "CL"): 1396,
                      ("dev", "IT"): 746, ("test", "CL"): 1339, ("test", "IT"): 858}

ID_KEYS = ("id", "doc_id", "docid", "document_id", "documentid", "doc", "name", "file", "filename",
           "file_name", "case_id", "caseid", "doc_name")
SENT_KEYS = ("sentences", "sents", "sentence", "texts", "text", "segments", "lines")
LABEL_KEYS = ("labels", "label", "roles", "role", "rhetorical_roles", "rhetorical_role", "rr", "tags",
              "gold", "y")
DOMAIN_KEYS = ("domain", "category", "area", "subset")
CONTAINER_KEYS = ("data", "documents", "docs", "items", "records", "examples")


class Phase3FormatError(RuntimeError):
    pass


def _tokens(path):
    return [t for t in re.split(r"[^a-z0-9]+", path.lower()) if t]


def _split_hits(text):
    hits = set()
    for t in _tokens(text):
        if t.startswith("train"):
            hits.add("train")
        elif t in ("dev", "val", "valid", "validation", "development") or t.startswith("dev"):
            hits.add("dev")
        elif t.startswith("test"):
            hits.add("test")
    return hits


def split_from_path(path):
    """The file name decides first (Train/CL_dev.json is dev), then the folders, innermost first."""
    parts = path.replace("\\", "/").split("/")
    for part in [os.path.splitext(parts[-1])[0]] + parts[-2::-1]:
        hits = _split_hits(part)
        if len(hits) == 1:
            return hits.pop()
        if len(hits) > 1:
            return None
    return None


def domain_from_text(s):
    toks = _tokens(str(s))
    hits = set()
    for t in toks:
        if t in ("cl", "corporate", "corporatelaw", "company") or t.startswith("cl") and t[2:].isdigit():
            hits.add("CL")
        elif t in ("it", "infotech", "informationtechnology") or t.startswith("it") and t[2:].isdigit():
            hits.add("IT")
    return hits.pop() if len(hits) == 1 else None


def _first(d, keys):
    low = {k.lower(): k for k in d}
    for k in keys:
        if k in low:
            return d[low[k]]
    return None


def _is_label_list(v):
    return isinstance(v, list) and v and all(isinstance(x, str) for x in v) and all(x in LABEL_SET for x in v)


def _norm_label(x):
    if isinstance(x, list) and len(x) == 1:
        x = x[0]
    if isinstance(x, dict):
        x = _first(x, LABEL_KEYS)
        return _norm_label(x)
    return x


def _parse_sentence_dicts(items):
    sents, labs = [], []
    for it in items:
        s = _first(it, ("text", "sentence", "sent", "content", "line"))
        if s is None and "value" in it and isinstance(it["value"], dict):         # legal-eval result item
            s = it["value"].get("text")
            labs.append(_norm_label(it["value"].get("labels")))
        else:
            labs.append(_norm_label(_first(it, LABEL_KEYS)))
        sents.append(s)
    labs = None if all(l is None for l in labs) else labs
    return sents, labs


def parse_record(r, default_id=None):
    """-> (doc_id, sentences or None, labels or None, domain or None)"""
    did = _first(r, ID_KEYS)
    did = default_id if did is None else did
    dom = _first(r, DOMAIN_KEYS)
    dom = domain_from_text(dom) if dom is not None else None
    sents = labs = None
    ann = r.get("annotations")
    if isinstance(ann, list) and ann and isinstance(ann[0], dict) and isinstance(ann[0].get("result"), list):
        sents, labs = _parse_sentence_dicts(ann[0]["result"])
    else:
        sv = _first(r, SENT_KEYS)
        if isinstance(sv, dict):                                      # e.g. {"text": {...}} nested
            return parse_record({**sv, **{k: v for k, v in r.items() if k not in SENT_KEYS}}, did)
        if isinstance(sv, list) and sv and all(isinstance(x, dict) for x in sv):
            sents, labs = _parse_sentence_dicts(sv)
        elif isinstance(sv, list) and all(isinstance(x, str) for x in sv):
            sents = sv
        lv = _first(r, LABEL_KEYS)
        if isinstance(lv, list):
            labs = [_norm_label(x) for x in lv]
    if sents is not None and _is_label_list(sents) and labs is None:      # a truth file keyed under "text"
        sents, labs = None, sents
    return (None if did is None else str(did)), sents, labs, dom


def parse_obj(obj):
    out = []
    if isinstance(obj, dict):
        cont = _first(obj, CONTAINER_KEYS)
        if isinstance(cont, list) and len(obj) <= 3:
            return parse_obj(cont)
        for k, v in obj.items():
            if isinstance(v, list) and all(isinstance(x, str) for x in v):
                out.append((str(k), None, v, None) if _is_label_list(v) else (str(k), v, None, None))
            elif isinstance(v, list) and v and all(isinstance(x, dict) for x in v):
                s, l = _parse_sentence_dicts(v)
                out.append((str(k), s, l, None))
            elif isinstance(v, dict):
                out.append(parse_record(v, default_id=k))
            elif isinstance(v, list) and not v:
                out.append((str(k), [], None, None))
            else:
                raise Phase3FormatError(f"unrecognised value type for key {k!r}: {type(v).__name__}")
    elif isinstance(obj, list):
        for i, r in enumerate(obj):
            if not isinstance(r, dict):
                raise Phase3FormatError(f"list item {i} is {type(r).__name__}, expected an object")
            out.append(parse_record(r))
    else:
        raise Phase3FormatError(f"top-level JSON is {type(obj).__name__}")
    return out


def parse_csv(path):
    with open(path, encoding="utf-8", newline="") as f:
        sample = f.read(4096)
        f.seek(0)
        dialect = csv.Sniffer().sniff(sample, delimiters=",\t;|")
        rows = list(csv.DictReader(f, dialect=dialect))
    if not rows:
        return []
    cols = {c.lower(): c for c in rows[0]}
    idc = next((cols[k] for k in ID_KEYS if k in cols), None)
    sc = next((cols[k] for k in ("sentence", "text", "sent", "sentences") if k in cols), None)
    lc = next((cols[k] for k in LABEL_KEYS if k in cols), None)
    if idc is None or sc is None:
        raise Phase3FormatError(f"{path}: CSV needs a document-id and a sentence column, got {list(rows[0])}")
    docs = defaultdict(lambda: ([], []))
    order = []
    for r in rows:
        d = str(r[idc])
        if d not in docs:
            order.append(d)
        docs[d][0].append(r[sc])
        docs[d][1].append(r[lc] if lc else None)
    return [(d, docs[d][0], docs[d][1] if lc else None, None) for d in order]


def read_any(path):
    ext = os.path.splitext(path)[1].lower()
    if ext == ".jsonl":
        recs = [json.loads(l) for l in open(path, encoding="utf-8") if l.strip()]
        return parse_obj(recs)
    if ext == ".json":
        return parse_obj(json.load(open(path, encoding="utf-8")))
    if ext in (".csv", ".tsv"):
        return parse_csv(path)
    return None


def find_label_file(data_dir):
    for dp, _, fn in os.walk(data_dir):
        for f in fn:
            if "label" in f.lower() and os.path.splitext(f)[1].lower() in (".json", ".txt"):
                p = os.path.join(dp, f)
                try:
                    if f.lower().endswith(".txt"):
                        labs = [x.strip() for x in open(p, encoding="utf-8") if x.strip()]
                    else:
                        d = json.load(open(p, encoding="utf-8"))
                        if isinstance(d, list):
                            labs = d
                        elif isinstance(d, dict) and all(isinstance(v, int) for v in d.values()):
                            labs = list(d)
                        elif isinstance(d, dict):
                            labs = list(d.values())
                        else:
                            continue
                    if len(labs) == 13 and all(isinstance(x, str) for x in labs):
                        return p, labs
                except Exception:
                    continue
    return None, None


def inventory(data_dir, max_preview=300):
    """Human-readable listing of the dataset folder (for the notebook's inspection cell)."""
    lines = []
    for dp, _, fn in sorted(os.walk(data_dir)):
        for f in sorted(fn):
            p = os.path.join(dp, f)
            rel = os.path.relpath(p, data_dir)
            lines.append(f"{rel}  ({os.path.getsize(p) / 1024:.1f} KB)  split={split_from_path(rel)} "
                         f"domain={domain_from_text(rel)}")
            if f.lower().endswith((".json", ".jsonl", ".csv", ".tsv", ".txt")):
                with open(p, encoding="utf-8", errors="replace") as fh:
                    head = fh.read(max_preview).replace("\n", "\\n")
                lines.append(f"    preview: {head}")
    return "\n".join(lines)


def load_phase3(data_dir, seed=42, dev_docs_per_domain_fallback=5, verbose=True, allow_dev_fallback=False):
    """-> docs: list of dict(id, split, domain, sentences, labels|None), label_file_path|None"""
    label_path, file_labels = find_label_file(data_dir)
    if file_labels is not None and set(file_labels) != LABEL_SET:
        raise Phase3FormatError(f"label file {label_path} differs from the official 13 labels: {file_labels}")

    merged = {}                                      # (split, domain, id) -> dict
    used_files = []
    for dp, _, fn in sorted(os.walk(data_dir)):
        for f in sorted(fn):
            p = os.path.join(dp, f)
            if p == label_path:
                continue
            rel = os.path.relpath(p, data_dir)
            try:
                parsed = read_any(p)
            except Phase3FormatError as e:
                raise Phase3FormatError(f"{rel}: {e}")
            except (json.JSONDecodeError, UnicodeDecodeError, csv.Error) as e:
                if verbose:
                    print(f"  skip {rel}: not parseable ({type(e).__name__})")
                continue
            if parsed is None:
                continue
            split = split_from_path(rel)
            fdom = domain_from_text(rel)
            if split is None:
                print(f"  skip {rel}: cannot tell train/dev/test from its path")
                continue
            used_files.append(rel)
            for did, sents, labs, rdom in parsed:
                dom = rdom or fdom
                if did is None:
                    raise Phase3FormatError(f"{rel}: record without a document id")
                if dom is None:
                    raise Phase3FormatError(f"{rel}: cannot tell CL/IT for doc {did} (no domain in path or record)")
                key = (split, dom, did)
                cur = merged.setdefault(key, {"id": did, "split": split, "domain": dom,
                                              "sentences": None, "labels": None, "src": []})
                cur["src"].append(rel)
                if sents is not None:
                    if cur["sentences"] is not None and cur["sentences"] != sents:
                        raise Phase3FormatError(f"doc {did}: two different sentence lists ({cur['src']})")
                    cur["sentences"] = sents
                if labs is not None:
                    if cur["labels"] is not None and cur["labels"] != labs:
                        raise Phase3FormatError(f"doc {did}: two different label lists ({cur['src']})")
                    cur["labels"] = labs
    if not merged:
        raise Phase3FormatError(f"no documents parsed under {data_dir}")

    docs = sorted(merged.values(), key=lambda d: (d["split"], d["domain"], d["id"]))
    for d in docs:
        if d["sentences"] is None:
            raise Phase3FormatError(f"{d['split']}/{d['domain']}/{d['id']}: labels found but no sentences ({d['src']})")
        d["sentences"] = ["" if s is None else str(s) for s in d["sentences"]]
        if d["split"] == "test":
            if d["labels"] is not None:
                print(f"  WARNING: test doc {d['id']} carries labels - they are DROPPED and never used")
            d["labels"] = None
            continue
        if d["labels"] is None:
            raise Phase3FormatError(f"{d['split']}/{d['domain']}/{d['id']}: no labels for a {d['split']} document")
        if len(d["labels"]) != len(d["sentences"]):
            raise Phase3FormatError(f"{d['id']}: {len(d['sentences'])} sentences vs {len(d['labels'])} labels")
        bad = [l for l in d["labels"] if l not in LABEL_SET]
        if bad:
            raise Phase3FormatError(f"{d['id']}: unknown labels {sorted(set(map(str, bad)))[:5]}")

    have = {(d["split"], d["domain"]) for d in docs}
    if not any(s == "dev" for s, _ in have):
        if not allow_dev_fallback:
            raise Phase3FormatError("no dev split found - refusing to invent one from training documents "
                                    "(pass allow_dev_fallback=True only if the dataset truly has no dev files)")
        print(f"  WARNING: no dev split found -> holding out {dev_docs_per_domain_fallback} train docs per domain")
        rng = random.Random(seed)
        for dom in DOMAINS:
            ids = sorted(d["id"] for d in docs if d["split"] == "train" and d["domain"] == dom)
            rng.shuffle(ids)
            hold = set(ids[:dev_docs_per_domain_fallback])
            for d in docs:
                if d["split"] == "train" and d["domain"] == dom and d["id"] in hold:
                    d["split"] = "dev"
    ids_by_split = defaultdict(list)
    for d in docs:
        ids_by_split[d["split"]].append(d["id"])
    for s, ids in ids_by_split.items():
        norm = [i[:-4] if i.endswith(".txt") else i for i in ids]
        dup = [i for i, c in Counter(norm).items() if c > 1]
        if dup:
            raise Phase3FormatError(f"{s}: document ids repeat across CL/IT (the scorer would reject this): {dup[:5]}")
    if verbose:
        print("  files used:", used_files)
        print("  label file:", label_path)
    return docs, label_path


def describe(docs):
    rows = []
    for split in ("train", "dev", "test"):
        for dom in DOMAINS:
            ds = [d for d in docs if d["split"] == split and d["domain"] == dom]
            n = sum(len(d["sentences"]) for d in ds)
            exp = EXPECTED_SENTENCES.get((split, dom))
            flag = "" if exp is None or exp == n else f"   <-- brief says {exp}"
            rows.append(f"  {split:<5} {dom}: {len(ds):>3} docs {n:>6} sentences{flag}")
    print("\n".join(rows))
    for split in ("train", "dev"):
        c = Counter(l for d in docs if d["split"] == split for l in d["labels"])
        print(f"  {split} label counts:", {l: c.get(l, 0) for l in LABELS})
        missing = [l for l in LABELS if c.get(l, 0) == 0]
        if missing:
            print(f"  NOTE: labels with zero {split} support: {missing}"
                  + ("  -> each scores F1=0 in official dev Macro-F1 whatever the model does" if split == "dev" else ""))
'''

RR_TRAIN_SRC = r'''"""CS781 Phase 3 - Rhetorical Role prediction on top of the Phase-1 LegalLM.

One script, one experiment per call, one output directory per experiment:
    python rr_train.py --name e1_full --method full
    python rr_train.py --name e1_lorma --method lorma --hp '{"lr": 1e-3}'
    python rr_train.py --name smoke_full --method full --smoke        (2 docs/domain, a few steps)

Model ("packed-context hierarchical encoder"):
  consecutive sentences of a document are packed into one window (<= `window` tokens, an EOS between
  sentences) -> Phase-1 LegalLM with the causal mask removed (bidirectional=True), so every token sees
  the neighbouring sentences -> mean-pool each sentence's tokens -> + document-position bucket
  embedding + domain (CL/IT) embedding -> BiGRU over the sentences of the window -> 13-way softmax.
  Inference: overlapping windows (50% sentence overlap), centre-weighted average of probabilities.

Selection: official dev Macro-F1 from YOUR evaluate_phase3.py (run as a subprocess every epoch);
the in-script sklearn mirror must agree with it to 1e-6 or the run stops.
Test labels are never read. Submission: <out>/submission_<name>.zip with prediction.json at its root.
"""
import argparse
import hashlib
import json
import math
import os
import random
import re
import shutil
import subprocess
import sys
import time
import zipfile

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from rr_data import LABELS, DOMAINS, load_phase3, describe  # noqa: E402

L2I = {l: i for i, l in enumerate(LABELS)}
N_LABELS = len(LABELS)
DOM2I = {d: i for i, d in enumerate(DOMAINS)}

DEFAULTS = dict(
    bidirectional=True,       # drop the causal mask (Phase 2: bidirectional beat causal on this model)
    pooling="mean",           # "mean" | "last" (last is the natural choice when bidirectional=False)
    window=1024,              # tokens per packed window (Phase-1 max_seq_len = 1024)
    max_sent_tokens=256,      # a single sentence is truncated to this many tokens
    context="gru",            # "gru" | "none"   (BiGRU over the sentences of a window)
    gru_hidden=256,
    pos_buckets=16,           # document-position buckets (0 = off)
    domain_emb=True,
    epochs=12, patience=4, batch_size=4, eval_batch_size=8,
    lr=3e-5,                  # encoder (or adapter) learning rate
    head_lr=1e-3,             # pooling extras, GRU, classifier
    weight_decay=0.01, warmup=0.06, grad_clip=1.0, dropout=0.1,
    loss="ce",                # "ce" | "sqrt_weighted" (sqrt inverse-frequency, mean 1, clipped)
    label_smoothing=0.0,
    fp16=True, seed=42,
    max_steps_per_epoch=0,    # 0 = full epoch (smoke tests set this)
)
METHOD_DEFAULTS = {
    "full":   dict(lr=3e-5),
    "lora":   dict(lr=5e-4, rank=16, alpha=32, targets=["qkv", "proj", "w12", "w3"]),
    "rslora": dict(lr=5e-4, rank=16, alpha=32, targets=["qkv", "proj", "w12", "w3"]),
    # LoRMA diverged at lr 1e-3 with 1024-token windows (non-finite losses in epoch 2) -> 2e-4, fp32 update path
    "lorma":  dict(lr=2e-4, rank=16, alpha=32, targets=["qkv", "proj", "w12", "w3"]),
    "ia3":    dict(lr=3e-3),
    "frozen": dict(lr=0.0),   # linear-probe control: encoder frozen, only GRU/head train
    # knowledge distillation: smaller student (subset of Phase-1 layers, fully fine-tuned) learns from the
    # gold labels and from a finished teacher run's sentence-level probabilities (same windows, same batch)
    "kd":     dict(lr=5e-5, teacher="", student_layers=None, kd_alpha=0.5, kd_T=2.0),
}


def log(*a):
    print(*a, flush=True)


# ============================================================================ encoder (= Phase-1 weights)
class RMSNorm(nn.Module):
    def __init__(self, dim, eps):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        xf = x.float()
        return (xf * torch.rsqrt(xf.pow(2).mean(-1, keepdim=True) + self.eps) * self.weight).to(x.dtype)


def rope_tables(cfg):
    hd = cfg["d_model"] // cfg["n_head"]
    inv = 1.0 / (cfg.get("rope_theta", 10000.0) ** (torch.arange(0, hd, 2).float() / hd))
    f = torch.outer(torch.arange(cfg["max_seq_len"]).float(), inv)
    return f.cos(), f.sin()


def apply_rope(x, cos, sin):
    h = x.shape[-1] // 2
    x1, x2 = x[..., :h], x[..., h:]
    cos, sin = cos.to(x.dtype), sin.to(x.dtype)
    return torch.cat([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1)


class Attention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.h, self.hd = cfg["n_head"], cfg["d_model"] // cfg["n_head"]
        self.qkv = nn.Linear(cfg["d_model"], 3 * cfg["d_model"], bias=False)
        self.proj = nn.Linear(cfg["d_model"], cfg["d_model"], bias=False)

    def forward(self, x, cos, sin, mask, causal):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=2)
        q, k, v = (t.view(B, T, self.h, self.hd).transpose(1, 2) for t in (q, k, v))
        q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)
        if causal:
            y = F.scaled_dot_product_attention(q, k, v, is_causal=True)   # right padding never reaches real tokens
        else:
            y = F.scaled_dot_product_attention(q, k, v, attn_mask=mask)   # (B,1,1,T) True = attend
        return self.proj(y.transpose(1, 2).reshape(B, T, C))


class SwiGLU(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.w12 = nn.Linear(cfg["d_model"], 2 * cfg["d_ff"], bias=False)
        self.w3 = nn.Linear(cfg["d_ff"], cfg["d_model"], bias=False)

    def forward(self, x):
        a, b = self.w12(x).chunk(2, dim=-1)
        return self.w3(F.silu(a) * b)


class Block(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.norm1 = RMSNorm(cfg["d_model"], cfg.get("norm_eps", 1e-5))
        self.attn = Attention(cfg)
        self.norm2 = RMSNorm(cfg["d_model"], cfg.get("norm_eps", 1e-5))
        self.mlp = SwiGLU(cfg)

    def forward(self, x, cos, sin, mask, causal):
        x = x + self.attn(self.norm1(x), cos, sin, mask, causal)
        return x + self.mlp(self.norm2(x))


class LegalEncoder(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok = nn.Embedding(cfg["vocab_size"], cfg["d_model"])
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg["n_layer"])])
        self.norm = RMSNorm(cfg["d_model"], cfg.get("norm_eps", 1e-5))
        cos, sin = rope_tables(cfg)
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)

    def forward(self, ids, key_mask, causal):
        T = ids.shape[1]
        cos, sin = self.rope_cos[:T][None, None], self.rope_sin[:T][None, None]
        mask = None if causal else key_mask[:, None, None, :]
        x = self.tok(ids)
        for b in self.blocks:
            x = b(x, cos, sin, mask, causal)
        return self.norm(x)


# ============================================================================ PEFT modules
class LoRALinear(nn.Module):
    """W0 x + s * B A x ; s = alpha/r (LoRA) or alpha/sqrt(r) (rsLoRA). B = 0 at init."""
    def __init__(self, base, r, alpha, rs=False):
        super().__init__()
        self.base, self.s = base, (alpha / math.sqrt(r) if rs else alpha / r)
        self.A = nn.Parameter(torch.empty(r, base.in_features))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))

    def forward(self, x):
        return self.base(x) + self.s * F.linear(F.linear(x, self.A), self.B)


class LoRMALinear(nn.Module):
    """LoRMA+: (I + s B A) W0 x, computed as y0 + s B (A y0). B = 0 at init (same as your Phase 2 code)."""
    def __init__(self, base, r, alpha):
        super().__init__()
        self.base, self.s = base, alpha / r
        self.A = nn.Parameter(torch.empty(r, base.out_features))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))

    def forward(self, x):
        y0 = self.base(x)
        with torch.autocast("cuda", enabled=False):                       # low-rank update in fp32
            d = F.linear(F.linear(y0.float(), self.A.float()), self.B.float()) * self.s
        return y0 + d.to(y0.dtype)


class IA3QKV(nn.Module):
    """(IA)^3 on attention: rescale keys and values (queries untouched). Vectors start at 1."""
    def __init__(self, base, d):
        super().__init__()
        self.base = base
        self.l_k = nn.Parameter(torch.ones(d))
        self.l_v = nn.Parameter(torch.ones(d))
        self.register_buffer("ones_q", torch.ones(d), persistent=False)

    def forward(self, x):
        return self.base(x) * torch.cat([self.ones_q, self.l_k, self.l_v]).to(x.dtype)


class IA3In(nn.Module):
    """(IA)^3 on the FFN: rescale the d_ff activations entering w3."""
    def __init__(self, base):
        super().__init__()
        self.base = base
        self.l_ff = nn.Parameter(torch.ones(base.in_features))

    def forward(self, x):
        return self.base(x * self.l_ff.to(x.dtype))


def attach_peft(encoder, method, hp, d_model):
    n = 0
    for blk in encoder.blocks:
        if method == "ia3":
            blk.attn.qkv = IA3QKV(blk.attn.qkv, d_model)
            blk.mlp.w3 = IA3In(blk.mlp.w3)
            n += 2
            continue
        for parent, name in [(blk.attn, "qkv"), (blk.attn, "proj"), (blk.mlp, "w12"), (blk.mlp, "w3")]:
            if name not in hp["targets"]:
                continue
            base = getattr(parent, name)
            if method == "lora":
                mod = LoRALinear(base, hp["rank"], hp["alpha"])
            elif method == "rslora":
                mod = LoRALinear(base, hp["rank"], hp["alpha"], rs=True)
            elif method == "lorma":
                mod = LoRMALinear(base, hp["rank"], hp["alpha"])
            else:
                raise ValueError(method)
            setattr(parent, name, mod)
            n += 1
    return n


# ============================================================================ RR model
class RRModel(nn.Module):
    def __init__(self, cfg, hp):
        super().__init__()
        d = cfg["d_model"]
        self.hp = hp
        self.encoder = LegalEncoder(cfg)
        self.causal = not hp["bidirectional"]
        self.drop = nn.Dropout(hp["dropout"])
        self.pos_emb = nn.Embedding(hp["pos_buckets"], d) if hp["pos_buckets"] else None
        self.dom_emb = nn.Embedding(len(DOMAINS), d) if hp["domain_emb"] else None
        for e in (self.pos_emb, self.dom_emb):
            if e is not None:
                nn.init.normal_(e.weight, std=0.02)
        if hp["context"] == "gru":
            self.ctx = nn.GRU(d, hp["gru_hidden"], batch_first=True, bidirectional=True)
            out = d + 2 * hp["gru_hidden"]
        elif hp["context"] == "none":
            self.ctx, out = None, d
        else:
            raise ValueError(hp["context"])
        self.head = nn.Linear(out, N_LABELS)
        nn.init.normal_(self.head.weight, std=0.02)
        nn.init.zeros_(self.head.bias)

    def new_module_names(self):
        return ("pos_emb", "dom_emb", "ctx", "head")

    def forward(self, ids, key_mask, slot, n_sent, pos, dom):
        """ids/key_mask/slot: (B,T); slot = sentence index of each token, -1 for EOS/pad.
        n_sent: (B,) sentences per window; pos: (B,S) position bucket; dom: (B,)  ->  logits (B,S,13)"""
        h = self.encoder(ids, key_mask, self.causal).float()
        B, T, D = h.shape
        S = int(n_sent.max())
        valid = (slot >= 0)
        idx = slot.clamp(min=0)
        if self.hp["pooling"] == "mean":
            w = valid.float()
            sums = torch.zeros(B, S, D, device=h.device).scatter_add_(1, idx[..., None].expand(-1, -1, D), h * w[..., None])
            cnt = torch.zeros(B, S, device=h.device).scatter_add_(1, idx, w)
            v = sums / cnt.clamp(min=1.0)[..., None]
        elif self.hp["pooling"] == "last":
            tpos = torch.arange(T, device=h.device)[None].expand(B, T)
            lastpos = torch.full((B, S), -1, dtype=torch.long, device=h.device)
            lastpos.scatter_reduce_(1, idx, torch.where(valid, tpos, torch.full_like(tpos, -1)), reduce="amax")
            v = h.gather(1, lastpos.clamp(min=0)[..., None].expand(-1, -1, D))
        else:
            raise ValueError(self.hp["pooling"])
        if self.pos_emb is not None:
            v = v + self.pos_emb(pos)
        if self.dom_emb is not None:
            v = v + self.dom_emb(dom)[:, None]
        v = self.drop(v)
        if self.ctx is not None:
            packed = nn.utils.rnn.pack_padded_sequence(v, n_sent.cpu(), batch_first=True, enforce_sorted=False)
            c, _ = self.ctx(packed)
            c, _ = nn.utils.rnn.pad_packed_sequence(c, batch_first=True, total_length=S)
            v = torch.cat([v, self.drop(c)], -1)
        return self.head(v)


def load_phase1(model, model_dir, layer_map=None):
    sd = torch.load(os.path.join(model_dir, "pytorch_model.bin"), map_location="cpu")
    if isinstance(sd, dict) and "model" in sd and isinstance(sd["model"], dict):
        sd = sd["model"]                                          # a training checkpoint instead of final weights
    sd = {k[len("_orig_mod."):] if k.startswith("_orig_mod.") else k: v for k, v in sd.items()}
    new = {}
    for k, v in sd.items():
        if k.startswith("head."):
            continue                                              # LM head is tied to tok
        if k.startswith("blocks.") and layer_map is not None:     # KD student: keep a subset of layers
            i = int(k.split(".")[1])
            if i not in layer_map:
                continue
            k = f"blocks.{layer_map[i]}." + k.split(".", 2)[2]
        new["encoder." + k] = v
    missing, unexpected = model.load_state_dict(new, strict=False)
    new_mods = model.new_module_names()
    missing = [m for m in missing if not m.split(".")[0] in new_mods]
    if missing or unexpected:
        raise RuntimeError(f"Phase-1 weights do not match: missing={missing[:8]} unexpected={unexpected[:8]}")
    tied = torch.equal(sd["head.weight"], sd["tok.weight"]) if "head.weight" in sd else None
    return len(new), tied


def build_model(cfg, hp, method, model_dir):
    cfg = dict(cfg)
    layer_map = None
    if hp.get("student_layers") is not None or method == "kd":
        layers = list(hp.get("student_layers") or range(0, cfg["n_layer"], 2))   # default: every 2nd layer
        assert all(0 <= i < cfg["n_layer"] for i in layers) and len(set(layers)) == len(layers), layers
        hp["student_layers"] = layers
        layer_map = {src: dst for dst, src in enumerate(layers)}
        cfg["n_layer"] = len(layers)
    model = RRModel(cfg, hp)
    n_loaded, tied = load_phase1(model, model_dir, layer_map)
    if method not in ("full", "kd"):
        for p in model.encoder.parameters():
            p.requires_grad = False
    n_wrapped = attach_peft(model.encoder, method, hp, cfg["d_model"]) if method not in ("full", "frozen", "kd") else 0
    return model, n_loaded, tied, n_wrapped


def trainable_state(model):
    names = {n for n, p in model.named_parameters() if p.requires_grad}
    return {k: v.detach().cpu().clone() for k, v in model.state_dict().items() if k in names}


def load_trainable_state(model, state):
    names = {n for n, p in model.named_parameters() if p.requires_grad}
    if set(state) != names:
        raise RuntimeError(f"checkpoint/trainable mismatch: {sorted(set(state) ^ names)[:8]}")
    model.load_state_dict(state, strict=False)


# ============================================================================ data -> windows
class EncDoc:
    pass


def encode_docs(docs, tok, hp, eos_id):
    out = []
    for d in docs:
        e = EncDoc()
        e.id, e.domain, e.split = d["id"], d["domain"], d["split"]
        e.sent_ids = []
        for s in d["sentences"]:
            ids = tok.encode(s).ids[:hp["max_sent_tokens"]] if s.strip() else []
            e.sent_ids.append(ids if ids else [eos_id])          # empty sentence -> one placeholder token
        e.n = len(e.sent_ids)
        e.labels = np.array([L2I[l] for l in d["labels"]], dtype=np.int64) if d["labels"] is not None else None
        nb = hp["pos_buckets"] or 1
        e.pos = np.minimum((np.arange(e.n) * nb) // max(e.n, 1), nb - 1).astype(np.int64)
        out.append(e)
    return out


def pack_from(doc, start, budget):
    """Largest run of sentences starting at `start` that fits `budget` tokens (EOS after each sentence)."""
    used, end = 0, start
    while end < doc.n:
        need = len(doc.sent_ids[end]) + 1
        if end > start and used + need > budget:
            break
        used += need
        end += 1
    return end


def train_windows(docs, budget, rng):
    """Non-overlapping windows; random start offset per document and epoch -> varied context each epoch."""
    items = []
    for d in docs:
        first_end = pack_from(d, 0, budget)
        s = rng.randrange(0, max(1, first_end))
        if s > 0:
            items.append((d, 0, s))
        while s < d.n:
            e = pack_from(d, s, budget)
            items.append((d, s, e))
            s = e
    return items


def infer_windows(docs, budget):
    """Windows advance by half their sentence count -> every sentence is inside >= 1 window, mostly 2."""
    items = []
    for d in docs:
        s = 0
        while True:
            e = pack_from(d, s, budget)
            items.append((d, s, e))
            if e >= d.n:
                break
            s = max(s + 1, s + (e - s) // 2)
    return items


def collate(items, eos_id, with_labels=True):
    seqs, slots, ns = [], [], []
    for d, s, e in items:
        ids, sl = [], []
        for j, k in enumerate(range(s, e)):
            ids += d.sent_ids[k] + [eos_id]
            sl += [j] * len(d.sent_ids[k]) + [-1]
        seqs.append(ids); slots.append(sl); ns.append(e - s)
    B, T, S = len(items), max(map(len, seqs)), max(ns)
    ids = torch.full((B, T), eos_id, dtype=torch.long)
    km = torch.zeros((B, T), dtype=torch.bool)
    slot = torch.full((B, T), -1, dtype=torch.long)
    pos = torch.zeros((B, S), dtype=torch.long)
    lab = torch.full((B, S), -100, dtype=torch.long)
    dom = torch.tensor([DOM2I[d.domain] for d, _, _ in items], dtype=torch.long)
    for i, ((d, s, e), q, sl) in enumerate(zip(items, seqs, slots)):
        ids[i, :len(q)] = torch.tensor(q)
        km[i, :len(q)] = True
        slot[i, :len(sl)] = torch.tensor(sl)
        pos[i, :e - s] = torch.from_numpy(d.pos[s:e])
        if with_labels and d.labels is not None:
            lab[i, :e - s] = torch.from_numpy(d.labels[s:e])
    return ids, km, slot, torch.tensor(ns), pos, dom, lab


# ============================================================================ metrics + official evaluator
def sk_metrics(y_true, y_pred):
    from sklearn.metrics import f1_score, precision_score, recall_score, precision_recall_fscore_support, confusion_matrix
    m = dict(macro_f1=f1_score(y_true, y_pred, labels=LABELS, average="macro", zero_division=0),
             macro_p=precision_score(y_true, y_pred, labels=LABELS, average="macro", zero_division=0),
             macro_r=recall_score(y_true, y_pred, labels=LABELS, average="macro", zero_division=0))
    p, r, f, s = precision_recall_fscore_support(y_true, y_pred, labels=LABELS, zero_division=0)
    m["per_class"] = {l: dict(p=float(p[i]), r=float(r[i]), f1=float(f[i]), support=int(s[i])) for i, l in enumerate(LABELS)}
    m["confusion"] = confusion_matrix(y_true, y_pred, labels=LABELS).tolist()
    return m


def flatten(truth, pred, ids):
    yt, yp = [], []
    for d in ids:
        yt += truth[d]; yp += pred[d]
    return yt, yp


def full_report(dev_docs, preds):
    truth = {d.id: [LABELS[i] for i in d.labels] for d in dev_docs}
    rep = {"combined": sk_metrics(*flatten(truth, preds, list(truth)))}
    for dom in DOMAINS:
        ids = [d.id for d in dev_docs if d.domain == dom]
        if ids:
            rep[dom] = sk_metrics(*flatten(truth, preds, ids))
    return rep


class OfficialEvaluator:
    def __init__(self, evaluator_path, work_dir, dev_docs):
        self.path, self.dir = evaluator_path, work_dir
        os.makedirs(work_dir, exist_ok=True)
        self.labels_path = os.path.join(work_dir, "labels.json")
        json.dump(LABELS, open(self.labels_path, "w"))
        self.truth = {}
        for dom in DOMAINS:
            p = os.path.join(work_dir, f"dev_truth_{dom.lower()}.json")
            json.dump({d.id: [LABELS[i] for i in d.labels] for d in dev_docs if d.domain == dom},
                      open(p, "w"), ensure_ascii=False)
            self.truth[dom] = p

    def __call__(self, preds, tag):
        pp = os.path.join(self.dir, f"dev_prediction_{tag}.json")
        json.dump(preds, open(pp, "w"), ensure_ascii=False)
        cmd = [sys.executable, self.path, "--truth-cl", self.truth["CL"], "--truth-it", self.truth["IT"],
               "--prediction", pp, "--labels", self.labels_path]
        r = subprocess.run(cmd, capture_output=True, text=True)
        if r.returncode != 0:
            raise RuntimeError(f"official evaluator failed:\n{r.stdout}\n{r.stderr}")
        vals = {k: float(v) for k, v in re.findall(r"Macro-(F1|Precision|Recall):\s*([0-9.]+)", r.stdout)}
        if "F1" not in vals:
            raise RuntimeError(f"could not parse evaluator output:\n{r.stdout}")
        return vals, r.stdout


# ============================================================================ prediction
@torch.no_grad()
def predict(model, docs, hp, device, eos_id):
    model.eval()
    use_amp = hp["fp16"] and device.type == "cuda"
    items = infer_windows(docs, hp["window"])
    acc = {d.id: np.zeros((d.n, N_LABELS), dtype=np.float64) for d in docs}
    wsum = {d.id: np.zeros(d.n, dtype=np.float64) for d in docs}
    for k in range(0, len(items), hp["eval_batch_size"]):
        batch = items[k:k + hp["eval_batch_size"]]
        ids, km, slot, ns, pos, dom, _ = collate(batch, eos_id, with_labels=False)
        with torch.autocast("cuda", dtype=torch.float16, enabled=use_amp):
            logits = model(ids.to(device), km.to(device), slot.to(device), ns.to(device), pos.to(device), dom.to(device))
        probs = torch.softmax(logits.float(), -1).cpu().numpy()
        for i, (d, s, e) in enumerate(batch):
            j = np.arange(e - s)
            left = j if s > 0 else np.full(e - s, 10 ** 6)
            right = (e - s - 1) - j if e < d.n else np.full(e - s, 10 ** 6)
            w = np.minimum(np.minimum(left, right), 10 ** 3) + 1.0
            acc[d.id][s:e] += probs[i, :e - s] * w[:, None]
            wsum[d.id][s:e] += w
    preds, probs_out = {}, {}
    for d in docs:
        assert (wsum[d.id] > 0).all(), f"{d.id}: a sentence was not covered by any window"
        p = acc[d.id] / wsum[d.id][:, None]
        probs_out[d.id] = p
        preds[d.id] = [LABELS[i] for i in p.argmax(1)]
        assert len(preds[d.id]) == d.n
    return preds, probs_out


def dev_nll(dev_docs, probs):
    tot, n = 0.0, 0
    for d in dev_docs:
        p = probs[d.id][np.arange(d.n), d.labels]
        tot += -np.log(np.clip(p, 1e-12, 1)).sum(); n += d.n
    return tot / max(n, 1)


def class_weights(train_docs, hp):
    if hp["loss"] == "ce":
        return None
    if hp["loss"] != "sqrt_weighted":
        raise ValueError(hp["loss"])
    c = np.bincount(np.concatenate([d.labels for d in train_docs]), minlength=N_LABELS).astype(float)
    w = np.where(c > 0, np.sqrt(c.sum() / np.maximum(c, 1)), 1.0)
    w = w / (w * c).sum() * c.sum()                     # sentence-weighted mean = 1
    return torch.tensor(np.clip(w, 0.25, 10.0), dtype=torch.float32)


# ============================================================================ main
def param_fingerprint(params):
    h = hashlib.sha1()
    for p in params:
        h.update(p.detach().float().cpu().numpy().tobytes()[:4096])
    return h.hexdigest()


def write_submission(preds, test_docs, out_dir, name):
    clean = {}
    for d in test_docs:
        lab = preds[d.id]
        assert len(lab) == d.n and all(l in L2I for l in lab), d.id
        clean[d.id] = lab
    pj = os.path.join(out_dir, "prediction.json")
    json.dump(clean, open(pj, "w"), ensure_ascii=False)
    zp = os.path.join(out_dir, f"submission_{name}.zip")
    with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(pj, arcname="prediction.json")
    with zipfile.ZipFile(zp) as z:
        assert z.namelist() == ["prediction.json"]
    return zp, len(clean), sum(len(v) for v in clean.values())


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--name", required=True)
    ap.add_argument("--method", required=True, choices=sorted(METHOD_DEFAULTS))
    ap.add_argument("--hp", default="{}", help="JSON dict of overrides")
    ap.add_argument("--paths", default="/kaggle/working/rr_paths.json",
                    help="JSON with data_dir, model_dir, tokenizer, evaluator, out_root")
    ap.add_argument("--smoke", action="store_true")
    ap.add_argument("--force", action="store_true", help="re-run even if the experiment finished before")
    ap.add_argument("--model_dir", default=None, help="Phase-1 model folder for this run (default: paths.json)")
    ap.add_argument("--allow_count_mismatch", action="store_true",
                    help="train even if sentence counts differ from the brief")
    args = ap.parse_args()

    paths = json.load(open(args.paths))
    if args.model_dir:
        paths["model_dir"] = args.model_dir
        if os.path.exists(os.path.join(args.model_dir, "tokenizer.json")):
            paths["tokenizer"] = os.path.join(args.model_dir, "tokenizer.json")
    hp = {**DEFAULTS, **METHOD_DEFAULTS[args.method], **json.loads(args.hp)}
    unknown = set(json.loads(args.hp)) - set(DEFAULTS) - set(METHOD_DEFAULTS[args.method])
    if unknown:
        raise SystemExit(f"unknown hp keys: {sorted(unknown)}")
    name = ("_smoke_" + args.name) if args.smoke else args.name
    out_dir = os.path.join(paths["out_root"], name)
    if args.smoke and os.path.isdir(out_dir):
        shutil.rmtree(out_dir)
    if os.path.exists(os.path.join(out_dir, "DONE")) and not args.force:
        log(f"{name} already finished ({out_dir}); use a new --name, or --force to redo it")
        return
    os.makedirs(out_dir, exist_ok=True)

    random.seed(hp["seed"]); np.random.seed(hp["seed"]); torch.manual_seed(hp["seed"])
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if device.type == "cuda":
        log(f"[{name}] GPU {torch.cuda.get_device_name(0)} | visible GPUs {torch.cuda.device_count()} | "
            f"CUDA_VISIBLE_DEVICES={os.environ.get('CUDA_VISIBLE_DEVICES')}")
    else:
        log(f"[{name}] running on CPU")

    # ---- data
    from tokenizers import Tokenizer
    tok = Tokenizer.from_file(paths["tokenizer"])
    eos_id = tok.token_to_id("<|endoftext|>")
    assert eos_id is not None, "tokenizer has no <|endoftext|>"
    docs, _ = load_phase3(paths["data_dir"], seed=hp["seed"], verbose=False)
    if not args.smoke:
        from rr_data import EXPECTED_SENTENCES
        got = {k: sum(len(d["sentences"]) for d in docs if (d["split"], d["domain"]) == k) for k in EXPECTED_SENTENCES}
        bad = {k: (got[k], v) for k, v in EXPECTED_SENTENCES.items() if got[k] != v}
        if bad and not args.allow_count_mismatch:
            raise SystemExit(f"sentence counts differ from the brief (got, expected): {bad}")
    if args.smoke:
        keep = []
        for split in ("train", "dev", "test"):
            for dom in DOMAINS:
                keep += [d for d in docs if d["split"] == split and d["domain"] == dom][:2]
        docs = keep
        hp.update(epochs=2, patience=2, max_steps_per_epoch=10)
    describe(docs)
    enc = encode_docs(docs, tok, hp, eos_id)
    train_docs = [d for d in enc if d.split == "train"]
    dev_docs = [d for d in enc if d.split == "dev"]
    test_docs = [d for d in enc if d.split == "test"]
    tl = [len(s) for d in train_docs for s in d.sent_ids]
    log(f"sentence tokens (train): median {int(np.median(tl))}, p95 {int(np.percentile(tl, 95))}, max {max(tl)} "
        f"(cap {hp['max_sent_tokens']}); truncated {sum(t >= hp['max_sent_tokens'] for t in tl)}")

    # ---- model
    cfg = json.load(open(os.path.join(paths["model_dir"], "config.json")))
    model, n_loaded, tied, n_wrapped = build_model(cfg, hp, args.method, paths["model_dir"])
    model.to(device)
    tot = sum(p.numel() for p in model.parameters())
    trn = sum(p.numel() for p in model.parameters() if p.requires_grad)
    log(f"Phase-1 tensors loaded {n_loaded} (strict, LM head tied={tied}) | PEFT modules {n_wrapped} | "
        f"params {tot / 1e6:.2f}M, trainable {trn / 1e6:.3f}M ({100 * trn / tot:.2f}%)")
    frozen = [p for n, p in model.named_parameters() if not p.requires_grad]
    frozen_fp0 = param_fingerprint(frozen[:6]) if frozen else None
    json.dump({"name": name, "method": args.method, "hp": hp, "model_cfg": cfg, "paths": paths,
               "params_total": tot, "params_trainable": trn}, open(os.path.join(out_dir, "run_config.json"), "w"), indent=1)

    # ---- optimiser
    new_mods = model.new_module_names()
    enc_dec, enc_nodec, head = [], [], []
    for n, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if n.split(".")[0] in new_mods:
            head.append(p)
        elif p.ndim >= 2:
            enc_dec.append(p)
        else:
            enc_nodec.append(p)
    groups = [g for g in [{"params": enc_dec, "lr": hp["lr"], "weight_decay": hp["weight_decay"]},
                          {"params": enc_nodec, "lr": hp["lr"], "weight_decay": 0.0},
                          {"params": head, "lr": hp["head_lr"], "weight_decay": hp["weight_decay"]}] if g["params"]]
    opt = torch.optim.AdamW(groups)
    n0 = len(train_windows(train_docs, hp["window"], random.Random(0)))
    spe = math.ceil(n0 / hp["batch_size"])
    if hp["max_steps_per_epoch"]:
        spe = min(spe, hp["max_steps_per_epoch"])
    total = spe * hp["epochs"]
    warm = max(1, int(hp["warmup"] * total))
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: min(1.0, (s + 1) / warm) * max(0.0, min(1.0, (total - s) / max(1, total - warm))))
    use_amp = hp["fp16"] and device.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp, init_scale=2.0 ** 12, growth_interval=200)  # default 2^16 skips the first steps
    teacher, teacher_info = None, None
    if args.method == "kd":
        tdir = os.path.join(paths["out_root"], hp["teacher"])
        assert hp["teacher"] and os.path.exists(os.path.join(tdir, "DONE")), f"teacher run {tdir} has not finished"
        tmeta = json.load(open(os.path.join(tdir, "run_config.json")))
        thp = tmeta["hp"]
        for k in ("window", "max_sent_tokens", "pos_buckets"):
            assert thp[k] == hp[k], f"teacher/student must share {k}: {thp[k]} vs {hp[k]}"
        tmodel_dir = tmeta["paths"]["model_dir"]
        assert os.path.realpath(tmeta["paths"]["tokenizer"]) == os.path.realpath(paths["tokenizer"]) or \
            open(tmeta["paths"]["tokenizer"]).read() == open(paths["tokenizer"]).read(), "teacher uses another tokenizer"
        teacher, _, _, _ = build_model(tmeta["model_cfg"], thp, tmeta["method"], tmodel_dir)
        load_trainable_state(teacher, torch.load(os.path.join(tdir, "best.pt"), map_location="cpu"))
        teacher.to(device).eval()
        for p in teacher.parameters():
            p.requires_grad = False
        tm = json.load(open(os.path.join(tdir, "metrics.json")))
        teacher_info = {"name": hp["teacher"], "method": tmeta["method"], "params": sum(p.numel() for p in teacher.parameters()),
                        "layers": tmeta["model_cfg"]["n_layer"], "dev_f1_official": tm["dev_f1_official"]}
        log(f"teacher {teacher_info} | student layers {hp['student_layers']} "
            f"({len(hp['student_layers'])} of {cfg['n_layer']}) | kd_alpha {hp['kd_alpha']} T {hp['kd_T']}")

    cw = class_weights(train_docs, hp)
    cw = cw.to(device) if cw is not None else None
    log(f"windows/epoch ~{n0} | steps/epoch {spe} | total {total} | warmup {warm} | class weights "
        f"{None if cw is None else [round(x, 2) for x in cw.tolist()]}")

    ev = OfficialEvaluator(paths["evaluator"], os.path.join(out_dir, "eval"), dev_docs)

    # ---- resume
    start_ep, history, best, bad = 1, [], None, 0
    last_p, best_p = os.path.join(out_dir, "last.pt"), os.path.join(out_dir, "best.pt")
    if os.path.exists(last_p):
        ck = torch.load(last_p, map_location="cpu", weights_only=False)
        load_trainable_state(model, ck["model"])
        opt.load_state_dict(ck["opt"]); sched.load_state_dict(ck["sched"]); scaler.load_state_dict(ck["scaler"])
        start_ep, history, best, bad = ck["epoch"] + 1, ck["history"], ck["best"], ck["bad"]
        random.setstate(ck["py_rng"]); np.random.set_state(ck["np_rng"]); torch.set_rng_state(ck["torch_rng"])
        log(f"RESUMED after epoch {ck['epoch']} (best so far {best and best['dev_f1_official']})")
        del ck

    rng = random.Random(hp["seed"] * 1000 + start_ep)
    trainable = [p for p in model.parameters() if p.requires_grad]
    adapter_checked = start_ep > 1                      # on resume the check already ran
    for ep in range(start_ep, hp["epochs"] + 1):
        if bad >= hp["patience"]:
            break
        model.train()
        items = train_windows(train_docs, hp["window"], rng)
        rng.shuffle(items)
        t0, tot_loss, nb, nonfinite, gnorms, opt_steps = time.time(), 0.0, 0, 0, [], 0
        for k in range(0, len(items), hp["batch_size"]):
            if hp["max_steps_per_epoch"] and nb >= hp["max_steps_per_epoch"]:
                break
            ids, km, slot, ns, pos, dom, lab = collate(items[k:k + hp["batch_size"]], eos_id)
            ids, km, slot, pos, dom, lab = (t.to(device) for t in (ids, km, slot, pos, dom, lab))
            with torch.autocast("cuda", dtype=torch.float16, enabled=use_amp):
                logits = model(ids, km, slot, ns.to(device), pos, dom)
            logits = logits.float()
            valid = lab != -100
            loss = F.cross_entropy(logits[valid], lab[valid], weight=cw, label_smoothing=hp["label_smoothing"])
            if teacher is not None:
                with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16, enabled=use_amp):
                    t_logits = teacher(ids, km, slot, ns.to(device), pos, dom)
                T = hp["kd_T"]
                kd = F.kl_div(F.log_softmax(logits[valid] / T, -1), F.softmax(t_logits.float()[valid] / T, -1),
                              reduction="batchmean") * T * T
                loss = (1 - hp["kd_alpha"]) * loss + hp["kd_alpha"] * kd
            opt.zero_grad(set_to_none=True)
            if not torch.isfinite(loss):
                nonfinite += 1
                if nonfinite == 1:
                    log(f"  non-finite loss at epoch {ep} step {nb + 1}: |logits| max "
                        f"{logits.detach().abs().nan_to_num(posinf=1e30).max().item():.3g}, lr {[g['lr'] for g in opt.param_groups]}")
                if nonfinite > 20:
                    raise RuntimeError("more than 20 non-finite losses in one epoch - stopping")
                continue
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            gn = torch.nn.utils.clip_grad_norm_(trainable, hp["grad_clip"])
            scale_before = scaler.get_scale()
            scaler.step(opt); scaler.update()
            if scaler.get_scale() >= scale_before:              # optimizer actually stepped
                sched.step()
                opt_steps += 1
            if torch.isfinite(gn):
                gnorms.append(float(gn))
            tot_loss += loss.item(); nb += 1
            if ep == start_ep and nb == 1:
                no_grad = [n for n, p in model.named_parameters() if p.requires_grad and p.grad is None]
                log(f"  first step: loss {loss.item():.4f} | grad-norm {float(gn):.3f}"
                    f"{' (fp16 loss-scale warm-up: this step is skipped, normal)' if not torch.isfinite(gn) else ''}"
                    f" | trainable params without grad: "
                    f"{len(no_grad)} {no_grad[:3]}")
        train_s = time.time() - t0
        if nb == 0:
            raise RuntimeError("no optimisation step was taken this epoch")

        t1 = time.time()
        preds, probs = predict(model, dev_docs, hp, device, eos_id)
        off, _ = ev(preds, f"ep{ep}")
        rep = full_report(dev_docs, preds)
        mine = rep["combined"]["macro_f1"]
        if abs(mine - off["F1"]) > 1e-5:
            raise RuntimeError(f"in-script Macro-F1 {mine:.6f} != official {off['F1']:.6f}")
        rec = {"epoch": ep, "train_loss": tot_loss / nb, "dev_nll": dev_nll(dev_docs, probs),
               "dev_f1_official": off["F1"], "dev_p": off["Precision"], "dev_r": off["Recall"],
               **{f"dev_f1_{dm}": rep[dm]["macro_f1"] for dm in DOMAINS if dm in rep},
               "grad_norm_mean": float(np.mean(gnorms)) if gnorms else None, "nonfinite_steps": nonfinite,
               "steps": nb, "opt_steps": opt_steps, "train_s": train_s, "eval_s": time.time() - t1,
               "lr_now": [g["lr"] for g in opt.param_groups]}
        improved = best is None or off["F1"] > best["dev_f1_official"]
        if improved:
            best, bad = dict(rec), 0
            torch.save(trainable_state(model), best_p)
            json.dump(preds, open(os.path.join(out_dir, "dev_predictions_best.json"), "w"), ensure_ascii=False)
            json.dump(rep, open(os.path.join(out_dir, "dev_report_best.json"), "w"), indent=1)
        else:
            bad += 1
        history.append(rec)
        log(f"  ep {ep:>2}/{hp['epochs']} loss {rec['train_loss']:.4f} dev-nll {rec['dev_nll']:.4f} | OFFICIAL dev "
            f"Macro-F1 {off['F1']:.4f} (P {off['Precision']:.3f} R {off['Recall']:.3f}) | CL {rec.get('dev_f1_CL', 0):.4f} "
            f"IT {rec.get('dev_f1_IT', 0):.4f}{'  *' if improved else ''}  [{train_s:.0f}s+{rec['eval_s']:.0f}s]")
        if opt_steps == 0:
            log(f"  WARNING: all {nb} steps of epoch {ep} were skipped by the fp16 loss scaler (overflow) - no update yet")
        if not adapter_checked and frozen_fp0 is not None and opt_steps > 0:
            adapter_checked = True
            assert param_fingerprint(frozen[:6]) == frozen_fp0, "frozen pretrained weights changed!"
            if args.method in ("lora", "rslora", "lorma"):
                bn = sum(float(m.B.detach().norm()) for m in model.modules() if isinstance(m, (LoRALinear, LoRMALinear)))
                assert bn > 0, "LoRA/LoRMA B matrices are still zero - adapters did not update"
                log(f"  check: frozen weights unchanged, adapter |B| sum = {bn:.4f}")
            elif args.method == "ia3":
                dv = sum(float((m.l_k.detach() - 1).abs().sum() + (m.l_v.detach() - 1).abs().sum()) for m in model.modules() if isinstance(m, IA3QKV))
                assert dv > 0, "IA3 vectors did not move"
                log(f"  check: frozen weights unchanged, IA3 |l-1| sum = {dv:.4f}")
        torch.save({"model": trainable_state(model), "opt": opt.state_dict(), "sched": sched.state_dict(),
                    "scaler": scaler.state_dict(), "epoch": ep, "history": history, "best": best, "bad": bad,
                    "py_rng": random.getstate(), "np_rng": np.random.get_state(), "torch_rng": torch.get_rng_state()},
                   last_p + ".tmp")
        os.replace(last_p + ".tmp", last_p)
        if bad >= hp["patience"]:
            log(f"  early stop: no official dev improvement for {hp['patience']} epochs")

    if frozen_fp0 is not None and not adapter_checked:
        raise RuntimeError("no optimizer step ever succeeded, so frozen/adapter checks never ran")

    # ---- best checkpoint -> dev report + test submission
    load_trainable_state(model, torch.load(best_p, map_location="cpu"))
    model.to(device)
    preds, _ = predict(model, dev_docs, hp, device, eos_id)
    off, off_txt = ev(preds, "best")
    if abs(off["F1"] - best["dev_f1_official"]) > 1e-6:      # fp16 kernels can flip a near-tie
        log(f"  NOTE: reloaded best checkpoint scores {off['F1']:.6f} vs {best['dev_f1_official']:.6f} during training")
    rep = json.load(open(os.path.join(out_dir, "dev_report_best.json")))
    t0 = time.time()
    test_preds, _ = predict(model, test_docs, hp, device, eos_id)
    zp, n_docs, n_sent = write_submission(test_preds, test_docs, out_dir, name)
    if n_docs != 10 and not args.smoke:
        log(f"  WARNING: submission has {n_docs} test documents (brief expects 10)")

    log("\n" + "=" * 90 + f"\n{name} | best epoch {best['epoch']} | OFFICIAL dev Macro-F1 {off['F1']:.4f}  "
        f"(P {off['Precision']:.4f} R {off['Recall']:.4f})")
    log("official evaluator output:\n" + off_txt.strip())
    for dm in ["combined"] + [d for d in DOMAINS if d in rep]:
        log(f"  {dm:<8} macro-F1 {rep[dm]['macro_f1']:.4f}  P {rep[dm]['macro_p']:.4f}  R {rep[dm]['macro_r']:.4f}")
    log(f"  {'label':<24}{'P':>7}{'R':>7}{'F1':>7}{'supp':>6} | CL F1  IT F1")
    for l in LABELS:
        c = rep["combined"]["per_class"][l]
        dom_f = "  ".join(f"{rep[d]['per_class'][l]['f1']:.3f}" if d in rep else "  -  " for d in DOMAINS)
        log(f"  {l:<24}{c['p']:>7.3f}{c['r']:>7.3f}{c['f1']:>7.3f}{c['support']:>6} | {dom_f}")
    log(f"  confusion (rows = gold, cols = predicted, order = {LABELS}):")
    for l, row in zip(LABELS, rep["combined"]["confusion"]):
        log(f"  {l[:12]:<12} " + " ".join(f"{x:>4}" for x in row))
    log(f"  submission: {zp}  ({n_docs} docs, {n_sent} sentences, prediction.json at zip root)")

    metrics = {"name": name, "method": args.method, "best_epoch": best["epoch"],
               "dev_f1_official": off["F1"], "dev_p_official": off["Precision"], "dev_r_official": off["Recall"],
               **{f"dev_f1_{d}": rep[d]["macro_f1"] for d in DOMAINS if d in rep},
               "per_class_f1": {l: rep["combined"]["per_class"][l]["f1"] for l in LABELS},
               "params_total": tot, "params_trainable": trn, "layers": len(model.encoder.blocks),
               "teacher": teacher_info, "model_dir": paths["model_dir"], "history": history, "hp": hp,
               "test_docs": n_docs, "test_sentences": n_sent, "test_infer_s": time.time() - t0,
               "peak_gpu_gb": torch.cuda.max_memory_allocated() / 2 ** 30 if device.type == "cuda" else None}
    json.dump(metrics, open(os.path.join(out_dir, "metrics.json"), "w"), indent=1)
    open(os.path.join(out_dir, "DONE"), "w").write(time.ctime())
    if args.smoke:
        log("SMOKE TEST PASSED")


if __name__ == "__main__":
    main()
'''

for _name, _src in (("rr_data.py", RR_DATA_SRC), ("rr_train.py", RR_TRAIN_SRC)):
    with open(os.path.join(CODE_DIR, _name), "w", encoding="utf-8") as _f:
        _f.write(_src)
    print("wrote", os.path.join(CODE_DIR, _name), f"({len(_src.splitlines())} lines)")

# ---- re-check the data with the fixed loader
sys.path.insert(0, CODE_DIR)
import rr_data; importlib.reload(rr_data)
docs, label_file = rr_data.load_phase3(DATA_DIR)
rr_data.describe(docs)
got = {k: sum(len(d["sentences"]) for d in docs if (d["split"], d["domain"]) == k) for k in rr_data.EXPECTED_SENTENCES}
assert got == rr_data.EXPECTED_SENTENCES, f"sentence counts differ from the brief: {got}"
print("\nOK: sentence counts match the brief for train / dev / test in both domains")

# ---- helpers
def run(name, method, hp=None, gpu=0, smoke=False):
    cmd = [sys.executable, os.path.join(CODE_DIR, "rr_train.py"), "--name", name, "--method", method,
           "--hp", json.dumps(hp or {}), "--paths", "/kaggle/working/rr_paths.json"] + (["--smoke"] if smoke else [])
    env = {**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu), "PYTHONUNBUFFERED": "1"}
    r = subprocess.run(cmd, env=env, capture_output=True, text=True)
    print(r.stdout[-6000:]); print(r.stderr[-4000:])
    assert r.returncode == 0, f"{name} failed"


def run_parallel(jobs, poll=120):
    """jobs: list of (name, method, hp, gpu[, model_dir]). One process per GPU, waits for all, streams epoch lines."""
    os.makedirs(os.path.join(OUT_ROOT, "logs"), exist_ok=True)
    procs = []
    for job in jobs:
        name, method, hp, gpu = job[:4]
        lp = os.path.join(OUT_ROOT, "logs", f"{name}.log")
        cmd = [sys.executable, os.path.join(CODE_DIR, "rr_train.py"), "--name", name, "--method", method,
               "--hp", json.dumps(hp), "--paths", "/kaggle/working/rr_paths.json"]
        if len(job) > 4 and job[4]:
            cmd += ["--model_dir", job[4]]
        env = {**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu), "PYTHONUNBUFFERED": "1"}
        procs.append((name, lp, subprocess.Popen(cmd, env=env, stdout=open(lp, "a"), stderr=subprocess.STDOUT)))
        print(f"started {name} on GPU {gpu} -> {lp}")
    seen = {n: 0 for n, _, _ in procs}
    while True:
        time.sleep(poll)
        for n, lp, p in procs:
            lines = open(lp).read().splitlines()
            for l in lines[seen[n]:]:
                if l.lstrip().startswith(("ep ", "first step", "check:", "early stop", "RESUMED", "Phase-1", "teacher", "non-finite", "Trace", "Error", "RuntimeError", "AssertionError")) or "already finished" in l:
                    print(f"[{n}] {l.strip()}")
            seen[n] = len(lines)
        if all(p.poll() is not None for _, _, p in procs):
            break
    for n, lp, p in procs:
        print(f"\n{'#' * 30} {n}: exit code {p.returncode}")
        txt = open(lp).read()
        i = txt.rfind("=" * 90)
        print(txt[i:] if i >= 0 else txt[-5000:])


ISOLATED = dict(window=1, context="none", pos_buckets=0, domain_emb=False, batch_size=32, eval_batch_size=128)


def compare():
    import numpy as np
    LABELS = rr_data.LABELS
    rows = [json.load(open(p)) for p in sorted(glob.glob(os.path.join(OUT_ROOT, "*", "metrics.json"))) if "/_smoke_" not in p and os.path.basename(os.path.dirname(p)).startswith("v2_")]
    hdr = (f"{'run':<20}{'method':<7}{'ep':>3}{'devF1':>8}{'P':>7}{'R':>7}{'CL F1':>7}{'IT F1':>7}{'layers':>7}"
           f"{'params':>9}{'trainable':>10}{'GPU GB':>7}{'test s':>7}  model")
    print(hdr); print("-" * len(hdr))
    for m in rows:
        print(f"{m['name']:<20}{m['method']:<7}{m['best_epoch']:>3}{m['dev_f1_official']:>8.4f}{m['dev_p_official']:>7.3f}"
              f"{m['dev_r_official']:>7.3f}{m.get('dev_f1_CL', float('nan')):>7.3f}{m.get('dev_f1_IT', float('nan')):>7.3f}"
              f"{m.get('layers', 0):>7}{m['params_total'] / 1e6:>8.1f}M{m['params_trainable'] / 1e6:>9.2f}M"
              f"{(m['peak_gpu_gb'] or 0):>7.2f}{m['test_infer_s']:>7.1f}  {os.path.basename(m.get('model_dir', ''))}")
    print("\nper-class dev F1:")
    print(f"{'label':<24}" + "".join(f"{m['name'][:14]:>15}" for m in rows))
    for l in LABELS:
        print(f"{l:<24}" + "".join(f"{m['per_class_f1'][l]:>15.3f}" for m in rows))
    print("\noverfitting view (train loss / dev NLL / official dev F1 per epoch):")
    for m in rows:
        print(m["name"], " | ".join(f"ep{h['epoch']} {h['train_loss']:.3f}/{h['dev_nll']:.3f}/{h['dev_f1_official']:.3f}" for h in m["history"]))


def download():
    import shutil, zipfile
    from IPython.display import FileLink, display
    os.chdir("/kaggle/working")
    done = sorted(p for p in glob.glob(os.path.join(OUT_ROOT, "*", "DONE")) if "/_smoke_" not in p and os.path.basename(os.path.dirname(p)).startswith("v2_"))
    with zipfile.ZipFile("rr_results_v2.zip", "w", zipfile.ZIP_DEFLATED) as z:
        for p in done:
            d = os.path.dirname(p); n = os.path.basename(d)
            sub = os.path.join(d, f"submission_{n}.zip")
            shutil.copy(sub, f"submission_{n}.zip"); display(FileLink(f"submission_{n}.zip"))
            for f in ("metrics.json", "dev_report_best.json", "run_config.json", "dev_predictions_best.json"):
                z.write(os.path.join(d, f), arcname=f"{n}/{f}")
            lp = os.path.join(OUT_ROOT, "logs", f"{n}.log")
            if os.path.exists(lp):
                z.write(lp, arcname=f"{n}/train.log")
    display(FileLink("rr_results_v2.zip"))


def free_v1_disk():
    """Delete last.pt of finished v1 runs (keeps their best.pt, reports and submissions)."""
    for p in glob.glob(os.path.join(OUT_ROOT, "*", "last.pt")):
        d = os.path.dirname(p)
        if not os.path.basename(d).startswith("v2_") and os.path.exists(os.path.join(d, "DONE")):
            print("removing", p, f"({os.path.getsize(p) / 2**30:.2f} GB)"); os.remove(p)

print("\nv2 setup done - now run the short cells: smoke tests, round 1, compare(), round 2, round 3, download()")


wrote /kaggle/working/rr_code/rr_data.py (361 lines)
wrote /kaggle/working/rr_code/rr_train.py (879 lines)
  files used: ['Test/CL_test.json', 'Test/IT_test.json', 'Train/CL_dev.json', 'Train/CL_train.json', 'Train/IT_dev.json', 'Train/IT_train.json']
  label file: /kaggle/working/hf_phase3/Phase_3_RR/Train/label_vocab.json
  train CL:  40 docs  10593 sentences
  train IT:  40 docs   6252 sentences
  dev   CL:   5 docs   1396 sentences
  dev   IT:   5 docs    746 sentences
  test  CL:   5 docs   1339 sentences
  test  IT:   5 docs    858 sentences
  train label counts: {'Fact': 4098, 'Issue': 107, 'ArgumentPetitioner': 1613, 'ArgumentRespondent': 1348, 'Statute': 1099, 'Dissent': 284, 'PrecedentReliedUpon': 1945, 'PrecedentNotReliedUpon': 569, 'PrecedentOverruled': 3, 'RulingByLowerCourt': 874, 'RatioOfTheDecision': 4301, 'RulingByPresentCourt': 307, 'None': 297}
  dev label counts: {'Fact': 444, 'Issue': 15, 'ArgumentPetitioner': 400, 'ArgumentRespondent': 272, 'Statute': 120, 'Dissen

In [16]:
# smoke tests (~5 min)
run("full", "full", smoke=True, gpu=0)
run("lorma", "lorma", smoke=True, gpu=1)
run("kd", "kd", {"teacher": "_smoke_full"}, smoke=True, gpu=0)

[_smoke_full] GPU Tesla T4 | visible GPUs 1 | CUDA_VISIBLE_DEVICES=0
  train CL:   2 docs     73 sentences   <-- brief says 10593
  train IT:   2 docs    215 sentences   <-- brief says 6252
  dev   CL:   2 docs    332 sentences   <-- brief says 1396
  dev   IT:   2 docs    168 sentences   <-- brief says 746
  test  CL:   2 docs    344 sentences   <-- brief says 1339
  test  IT:   2 docs    199 sentences   <-- brief says 858
  train label counts: {'Fact': 75, 'Issue': 8, 'ArgumentPetitioner': 17, 'ArgumentRespondent': 12, 'Statute': 44, 'Dissent': 0, 'PrecedentReliedUpon': 18, 'PrecedentNotReliedUpon': 7, 'PrecedentOverruled': 0, 'RulingByLowerCourt': 15, 'RatioOfTheDecision': 68, 'RulingByPresentCourt': 16, 'None': 8}
  NOTE: labels with zero train support: ['Dissent', 'PrecedentOverruled']
  dev label counts: {'Fact': 106, 'Issue': 5, 'ArgumentPetitioner': 58, 'ArgumentRespondent': 101, 'Statute': 10, 'Dissent': 0, 'PrecedentReliedUpon': 22, 'PrecedentNotReliedUpon': 0, 'PrecedentOver

In [17]:
# round 1: context vs sentence-only on the official dev set (~15-20 min)
run_parallel([("v2_ctx_full",  "full", {}, 0),
              ("v2_sent_full", "full", ISOLATED, 1)])
compare()

started v2_ctx_full on GPU 0 -> /kaggle/working/rr/logs/v2_ctx_full.log
started v2_sent_full on GPU 1 -> /kaggle/working/rr/logs/v2_sent_full.log
[v2_ctx_full] Phase-1 tensors loaded 74 (strict, LM head tied=True) | PEFT modules 0 | params 111.14M, trainable 111.136M (100.00%)
[v2_ctx_full] first step: loss 2.4729 | grad-norm 18.474 | trainable params without grad: 0 []
[v2_ctx_full] ep  1/12 loss 1.8762 dev-nll 1.4145 | OFFICIAL dev Macro-F1 0.3237 (P 0.360 R 0.321) | CL 0.3088 IT 0.2744  *  [57s+6s]
[v2_sent_full] Phase-1 tensors loaded 74 (strict, LM head tied=True) | PEFT modules 0 | params 109.54M, trainable 109.540M (100.00%)
[v2_sent_full] first step: loss 2.9276 | grad-norm 11.228 | trainable params without grad: 0 []
[v2_ctx_full] ep  2/12 loss 1.2501 dev-nll 1.4802 | OFFICIAL dev Macro-F1 0.2621 (P 0.507 R 0.260) | CL 0.2214 IT 0.2793  [63s+5s]
[v2_ctx_full] ep  3/12 loss 0.8018 dev-nll 1.4675 | OFFICIAL dev Macro-F1 0.3990 (P 0.407 R 0.425) | CL 0.3432 IT 0.4177  *  [63s+5s]

In [18]:
# round 2: LoRMA + knowledge distillation from v2_ctx_full (~15 min)
run_parallel([("v2_ctx_lorma", "lorma", {}, 0),
              ("v2_ctx_kd6",   "kd",    {"teacher": "v2_ctx_full"}, 1)])
compare()

started v2_ctx_lorma on GPU 0 -> /kaggle/working/rr/logs/v2_ctx_lorma.log
started v2_ctx_kd6 on GPU 1 -> /kaggle/working/rr/logs/v2_ctx_kd6.log
[v2_ctx_lorma] Phase-1 tensors loaded 74 (strict, LM head tied=True) | PEFT modules 48 | params 114.18M, trainable 4.654M (4.08%)
[v2_ctx_lorma] first step: loss 2.4729 | grad-norm 15.909 | trainable params without grad: 0 []
[v2_ctx_lorma] ep  1/12 loss 1.9102 dev-nll 1.6266 | OFFICIAL dev Macro-F1 0.2715 (P 0.378 R 0.263) | CL 0.2431 IT 0.2385  *  [69s+9s]
[v2_ctx_lorma] check: frozen weights unchanged, adapter |B| sum = 17.2151
[v2_ctx_kd6] Phase-1 tensors loaded 38 (strict, LM head tied=True) | PEFT modules 0 | params 68.66M, trainable 68.660M (100.00%)
[v2_ctx_kd6] teacher {'name': 'v2_ctx_full', 'method': 'full', 'params': 111136269, 'layers': 12, 'dev_f1_official': 0.423553} | student layers [0, 2, 4, 6, 8, 10] (6 of 12) | kd_alpha 0.5 T 2.0
[v2_ctx_kd6] first step: loss 4.7800 | grad-norm 29.811 | trainable params without grad: 0 []
[v2

In [19]:
# round 3: rsLoRA + IA3 (~15 min)
run_parallel([("v2_ctx_rslora", "rslora", {}, 0),
              ("v2_ctx_ia3",    "ia3",    {}, 1)])
compare()

started v2_ctx_rslora on GPU 0 -> /kaggle/working/rr/logs/v2_ctx_rslora.log
started v2_ctx_ia3 on GPU 1 -> /kaggle/working/rr/logs/v2_ctx_ia3.log
[v2_ctx_rslora] Phase-1 tensors loaded 74 (strict, LM head tied=True) | PEFT modules 48 | params 113.50M, trainable 3.966M (3.49%)
[v2_ctx_rslora] first step: loss 2.4729 | grad-norm 16.799 | trainable params without grad: 0 []
[v2_ctx_rslora] ep  1/12 loss 1.9056 dev-nll 1.5203 | OFFICIAL dev Macro-F1 0.3212 (P 0.424 R 0.318) | CL 0.2933 IT 0.2781  *  [54s+7s]
[v2_ctx_rslora] check: frozen weights unchanged, adapter |B| sum = 37.5597
[v2_ctx_ia3] Phase-1 tensors loaded 74 (strict, LM head tied=True) | PEFT modules 24 | params 111.18M, trainable 1.649M (1.48%)
[v2_ctx_ia3] first step: loss 2.4729 | grad-norm 15.646 | trainable params without grad: 0 []
[v2_ctx_ia3] ep  1/12 loss 1.9022 dev-nll 1.6957 | OFFICIAL dev Macro-F1 0.2517 (P 0.297 R 0.254) | CL 0.2364 IT 0.2043  *  [49s+6s]
[v2_ctx_ia3] check: frozen weights unchanged, IA3 |l-1| sum 

In [20]:
download()

/kaggle/working/submission_v2_ctx_full.zip

/kaggle/working/submission_v2_ctx_ia3.zip

/kaggle/working/submission_v2_ctx_kd6.zip

/kaggle/working/submission_v2_ctx_lorma.zip

/kaggle/working/submission_v2_ctx_rslora.zip

/kaggle/working/submission_v2_sent_full.zip

/kaggle/working/rr_results_v2.zip